# cinn-audit: attribution, sensitivity and validity of activity measurements in neural networks

**Hassan Ugail (University of Bradford) and Newton Howard (Rochester Institute of Technology)**

This notebook reproduces the two main experiments of the paper *Attribution, Sensitivity and Validity in the
Measurement of Neural Network Activity*. It is self-contained. The `cinn-audit` library is inlined in a cell
below, so nothing has to be installed beyond the packages pulled in by `pip` in the setup cell.

## The problem

Knowing how accurately a neural network performs a task does not tell us how its internal activity is organised,
how that organisation changes with training, or what happens when part of the network is altered. Measurements
borrowed from the analysis of brain recordings, such as temporal correlation exponents, synchronisation and
representational structure, are increasingly used to answer these questions. The difficulty is that the activity
they are computed from belongs to the network and to its input at the same time. A feedforward filter driven by
noise with long temporal correlations produces output with the same correlations, even though it has no dynamics
of its own. A recurrent network responds differently to different inputs, and the input can change the regime the
network is in. On top of that, the value obtained depends on which units are recorded, how they are sampled and
which analysis settings are used. A number on its own cannot say where a pattern came from or what it means.

## What the framework does

The framework asks three separate questions of any activity measurement $z = \mathcal{M}_\pi(\mathcal{S}_\theta[u])$,
where $\mathcal{S}_\theta$ is the system, $u$ the input and $\mathcal{M}_\pi$ the measurement with its settings.

| Question | Experiment | Output |
|---|---|---|
| **Attribution.** Where does the variation in $z$ come from? | A crossed design in which every system receives the same inputs, with repeated runs that differ only in internal noise. | Shares of variance for the system, the input, their interaction and the repeats, obtained as sequential increments in $R^2$ of the model $z_{ijk} = \mu + A_i + B_j + C_{ij} + \varepsilon_{ijk}$. |
| **Conditional sensitivity.** Does $z$ move when the system is changed and the input is not? | A paired comparison of the intact and an altered system on identical inputs. | Mean paired change, paired effect size $d_z$, sign consistency, a Wilcoxon test, and the change scaled by the between-input spread. |
| **Validity and persistence.** Does $z$ measure the property it is named after, and does the activity continue when the input is removed? | Checks matched to the claimed property, here scale-free organisation measured by the DFA exponent, plus a zero-input run. | A verdict per channel from linearity, crossover, spectral-peak, stationarity, range and phase-randomised surrogate checks, a majority rule per system, and a persistence verdict under a stated system boundary. |

The answers can differ for the same statistic. That is the point of asking them separately.

## What this notebook runs

| Section | Experiment | What it tests | Runtime |
|---|---|---|---|
| E2v2 | Six families of reference systems with known mechanisms (feedforward filters, leaky networks, chaotic rate networks swept across gain with the matrix held fixed, Stuart–Landau oscillators, heterogeneous-timescale networks, critical branching networks), 33 systems in all, driven by the same eight inputs and run on zero input. | Whether the validity checks accept scaling only where the mechanism produces it, how the variance shares differ between families, and whether unit removal and weight noise are detected by the same statistics. | CPU, about 25 minutes |
| E4v2 | Pythia-410M, 1.4B and 2.8B, each compared with a head-pruned and a weight-noised version of itself, with every generated text replayed through every variant. | How much of the variation in hidden-state statistics belongs to the text and how much to the model variant, and whether head pruning and loss-calibrated weight noise move those statistics consistently when the text is held fixed. | GPU (T4), about 40 minutes |

The names E2v2 and E4v2 are kept because the result folders and the figure code refer to them. The calibration and
training-trajectory experiments of the paper are smaller and are not part of this notebook.

## Outputs

Every result is written as a CSV or JSON file under `results/E2v2/` and `results/E4v2/` in the project directory,
and the whole results folder is bundled as `cinn_audit_results.zip` at the end. The main tables are

* `E2v2/driven_measurements.csv`, `E2v2/autonomous_measurements.csv`, `E2v2/attribution.csv`,
  `E2v2/inferential_profile.csv`, `E2v2/perturbation_measurements.csv`, `E2v2/conditional_sensitivity.csv`
* `E4v2/temporal_<model>.csv`, `E4v2/pairwise_<model>.csv`, `E4v2/calibration_<model>.json`,
  `E4v2/decomposition.csv`, `E4v2/conditional_sensitivity.csv`, `E4v2/permutation_nulls.csv`, `E4v2/severity.csv`

## How to run

1. Open the notebook in Google Colab with a GPU runtime (a T4 is sufficient) or in a local Jupyter with a CUDA GPU.
2. Set `PROJECT_DIR` in the **Paths** cell. It is the only place a directory is named. With `USE_DRIVE = True`
   the folder lives in your Google Drive and results persist between sessions. With `USE_DRIVE = False`
   everything is written to the local runtime.
3. Run all cells. Set `DRY_RUN = True` in the **Parameters** cell to check the whole pipeline on tiny stand-ins
   in a few minutes before committing to the full run.

The E4v2 cell skips generation and replay for any model whose result files already exist, so an interrupted
run can be resumed by running the notebook again.

## Paths

The only cell that names a directory.

In [ ]:
# ---- Paths: the only place directories are defined ------------------------------------------
import os

USE_DRIVE   = True                                   # False: work in the local runtime instead (nothing persists)
DRIVE_MOUNT = "/content/drive"
PROJECT_DIR = DRIVE_MOUNT + "/MyDrive/cinn-audit"    # <- change this line to relocate everything
LOCAL_DIR   = "/content/cinn_audit_work"             # used only when USE_DRIVE is False

if USE_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=False)
else:
    PROJECT_DIR = LOCAL_DIR

RESULTS_DIR = os.path.join(PROJECT_DIR, "results")   # results/E2v2 and results/E4v2 are created below
HF_CACHE    = os.path.join(PROJECT_DIR, "hf_cache")  # Hugging Face model cache, so models download once
ZIP_STEM    = os.path.join(PROJECT_DIR, "cinn_audit_results")

for d in (PROJECT_DIR, RESULTS_DIR, HF_CACHE):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.chdir(PROJECT_DIR)
print("Project directory :", PROJECT_DIR)
print("Results directory :", RESULTS_DIR)

## Parameters

All experimental settings in one place. `DRY_RUN = True` substitutes tiny stand-ins for a quick check of the pipeline.

In [ ]:
# ---- Parameters ----------------------------------------------------------------------------
DRY_RUN = False
SEED    = 0

# E2v2 attribution on reference systems
E2_FS            = 250.0            # nominal sampling rate for band definitions (1 step = 4 ms)
E2_STEPS         = 4000             # driven run length
E2_ZERO_STEPS    = 8000             # autonomous run length
E2_N_INPUTS      = 8
E2_REPLICATES    = 2                # internal-noise replicates per (system, input) cell, for the interaction term
E2_ZERO_REPEATS  = 3
E2_CHAOTIC_N     = 500
E2_CHAOTIC_GAINS = [0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
E2_CHAOTIC_SEEDS = [0, 1, 2]        # replicate matrices, each swept across every gain
E2_LEAKY_TAUS    = [5.0, 20.0, 80.0]
E2_BRANCHING_SIGMAS = [0.9, 0.98, 0.995]
E2_N_SURROGATES  = 20

# E4v2 Pythia scaling series
E4_MODELS      = ["EleutherAI/pythia-410m", "EleutherAI/pythia-1.4b", "EleutherAI/pythia-2.8b"]
E4_N_PROMPTS   = 10
E4_NEW_TOKENS  = 192
E4_TEMPERATURE = 1.0
E4_PRUNE_KEEP_FRACTION = 0.25       # heads kept in the last ceil(L/6) layers
E4_NOISE_FRACTION      = 0.3        # default weight-noise fraction of each matrix's SD; replaced by the calibrated value when E4_MATCH_SEVERITY is True
E4_FP16_ABOVE_PARAMS   = 1.0e10     # models larger than this run in float16; all three Pythia models run in float32 (11.2 GB for 2.8B)
E4_MATCH_SEVERITY      = True       # calibrate the weight-noise fraction per model so its NLL increase matches head pruning's
E4_REUSE_EXISTING      = True       # skip generation and replay for a model whose temporal_/pairwise_ files already exist in results/E4v2
E4_NOISE_FRACTIONS     = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6]   # candidates for the calibration
E4_CHANNELS_TEMPORAL, E4_CHANNELS_CKA = 32, 128
E4_N_PERM, E4_N_BOOT = 200, 200

if DRY_RUN:
    E2_STEPS, E2_ZERO_STEPS, E2_N_INPUTS, E2_REPLICATES, E2_ZERO_REPEATS = 1500, 3000, 3, 2, 1
    E2_CHAOTIC_N, E2_CHAOTIC_GAINS, E2_CHAOTIC_SEEDS = 200, [0.8, 2.0], [0, 1]
    E2_LEAKY_TAUS, E2_BRANCHING_SIGMAS, E2_N_SURROGATES = [20.0], [0.98], 5
    E4_MODELS, E4_N_PROMPTS, E4_NEW_TOKENS, E4_N_PERM, E4_N_BOOT = ["dry-run-neox"], 4, 96, 20, 20
print("DRY_RUN =", DRY_RUN)

## Setup

In [ ]:
!pip install -q statsmodels scikit-learn transformers accelerate
import os, time, json, shutil
import numpy, scipy, pandas, statsmodels, torch
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__,
      "| statsmodels", statsmodels.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## Library

The `cinn-audit` library, inlined so that the notebook is self-contained. It provides the estimators (DFA, spectral slope, metastability), the validity and persistence checks, the six reference systems, the attribution and conditional-sensitivity statistics and the permutation and bootstrap procedures.

In [ ]:
# === cinn-audit library (inlined.py) ===
from __future__ import annotations
import json, warnings, zlib
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Literal, Callable, Mapping, Sequence
import numpy as np
import pandas as pd
import networkx as nx
from scipy.signal import butter, filtfilt, hilbert, welch
from scipy.stats import friedmanchisquare, gaussian_kde, rankdata, wilcoxon


# ---- config.py ----
"""Configuration objects for every methodological choice in the audit.

Every estimator and component takes an explicit configuration and records it
in its output, so an audit report always states which DFA scale grid, which
H_eff mapping, which Lempel-Ziv normalisation and which gamma cap were used.
The published settings of the Biological Cybernetics paper are available as
named presets; the behaviour of the released notebooks, where it differs
from the paper, is available as a separate ``legacy`` preset so the
difference is documented in code rather than lost.
"""


from dataclasses import asdict, dataclass, field, replace
from typing import Literal

HeffMode = Literal["gaussian", "triangular", "per_subject"]
LZNorm = Literal["n_over_log2n", "c_over_n"]


@dataclass(frozen=True)
class DFAConfig:
    """Detrended fluctuation analysis settings.

    ``min_window`` and ``max_window_fraction`` define the scale grid. The
    scale grid is always ascending and is rejected if ``max_window`` falls
    below ``min_window``, which is what happens silently in the released
    notebook for series shorter than 64 samples.

    ``min_samples`` is the shortest series for which an exponent is returned
    without a warning. Below ``hard_min_samples`` the estimator returns NaN.
    """

    min_window: int = 16
    max_window_fraction: float = 0.25
    n_scales: int = 10
    detrend_order: int = 1
    min_samples: int = 256
    hard_min_samples: int = 64


@dataclass(frozen=True)
class PACConfig:
    """Theta-gamma phase-amplitude coupling settings (Tort modulation index)."""

    theta_band: tuple[float, float] = (4.0, 8.0)
    gamma_band: tuple[float, float] = (30.0, 80.0)
    n_bins: int = 36
    amplitude_guard: float = 1e-3
    n_surrogates: int = 50
    filter_order: int = 4


@dataclass(frozen=True)
class LZCConfig:
    """Broadband-envelope Lempel-Ziv complexity settings."""

    band: tuple[float, float] = (1.0, 40.0)
    normalisation: LZNorm = "n_over_log2n"
    filter_order: int = 4


@dataclass(frozen=True)
class MetastabilityConfig:
    """Kuramoto order-parameter metastability settings."""

    band: tuple[float, float] = (8.0, 13.0)
    filter_order: int = 4


@dataclass(frozen=True)
class HeffConfig:
    """Mapping from the channel-averaged DFA exponent to H_eff.

    ``gaussian``      H_eff = exp(-(a - H_opt)^2 / (2 sigma_H^2))          (paper Eq. 3)
    ``triangular``    H_eff = max(0, 1 - |a - H_opt| / alpha_range)        (paper Eq. 4)
    ``per_subject``   triangular with H_opt and alpha_range estimated per subject
                      from a reference condition (paper Section 4.2, real EEG)
    """

    mode: HeffMode = "gaussian"
    h_opt: float = 0.35
    sigma_h: float = 0.12
    alpha_range: float | None = None
    anchor: Literal["median", "kde_mode"] = "median"
    min_subject_range: float = 0.02


@dataclass(frozen=True)
class CompositeConfig:
    """Weights of the composite index, which sum to one."""

    w_h: float = 0.40
    w_d: float = 0.35
    w_m: float = 0.25
    lam: float = 1.0

    def __post_init__(self) -> None:
        total = self.w_h + self.w_d + self.w_m
        if abs(total - 1.0) > 1e-9:
            raise ValueError(f"composite weights must sum to 1, got {total:.6f}")


@dataclass(frozen=True)
class AuditConfig:
    """Complete configuration of one audit run."""

    name: str = "custom"
    fs: float = 250.0
    dfa: DFAConfig = field(default_factory=DFAConfig)
    pac: PACConfig = field(default_factory=PACConfig)
    lzc: LZCConfig = field(default_factory=LZCConfig)
    metastability: MetastabilityConfig = field(default_factory=MetastabilityConfig)
    heff: HeffConfig = field(default_factory=HeffConfig)
    composite: CompositeConfig = field(default_factory=CompositeConfig)

    def to_dict(self) -> dict:
        return asdict(self)

    def with_fs(self, fs: float) -> "AuditConfig":
        return replace(self, fs=fs)


# ---------------------------------------------------------------------------
# Presets
# ---------------------------------------------------------------------------

#: Settings stated in the Biological Cybernetics paper for the synthetic
#: ensemble: 250 Hz, 30-80 Hz gamma, Gaussian H_eff with H_opt = 0.35 and
#: sigma_H = 0.12, LZ normalised by n / log2 n, weights 0.40 / 0.35 / 0.25.
BIOLCYB2026_SYNTHETIC = AuditConfig(
    name="biolcyb2026.synthetic",
    fs=250.0,
    heff=HeffConfig(mode="gaussian", h_opt=0.35, sigma_h=0.12),
)

#: Settings stated in the paper for the two-channel Sleep-EDF analysis:
#: 100 Hz, gamma capped at 45 Hz, per-subject Wake-anchored triangular
#: H_eff with the median Wake exponent as anchor.
BIOLCYB2026_SLEEPEDF = AuditConfig(
    name="biolcyb2026.sleepedf",
    fs=100.0,
    pac=PACConfig(gamma_band=(30.0, 45.0)),
    heff=HeffConfig(mode="per_subject", anchor="median"),
)

#: What the released validation notebook computes, where it differs from
#: the paper: triangular H_eff normalised by the ensemble alpha range for
#: the synthetic Monte Carlo, c/n Lempel-Ziv normalisation inside D, and a
#: KDE-mode Wake anchor for real EEG. Kept so the discrepancy is testable.
LEGACY_NOTEBOOK_SYNTHETIC = AuditConfig(
    name="legacy.notebook.synthetic",
    fs=250.0,
    lzc=LZCConfig(normalisation="c_over_n"),
    heff=HeffConfig(mode="triangular", h_opt=0.35, sigma_h=0.12, alpha_range=None),
)

LEGACY_NOTEBOOK_SLEEPEDF = AuditConfig(
    name="legacy.notebook.sleepedf",
    fs=100.0,
    pac=PACConfig(gamma_band=(30.0, 45.0)),
    lzc=LZCConfig(normalisation="c_over_n"),
    heff=HeffConfig(mode="per_subject", anchor="kde_mode"),
)

PRESETS: dict[str, AuditConfig] = {
    c.name: c
    for c in (
        BIOLCYB2026_SYNTHETIC,
        BIOLCYB2026_SLEEPEDF,
        LEGACY_NOTEBOOK_SYNTHETIC,
        LEGACY_NOTEBOOK_SLEEPEDF,
    )
}


def get_preset(name: str) -> AuditConfig:
    try:
        return PRESETS[name]
    except KeyError as exc:
        raise KeyError(f"unknown preset {name!r}; available: {sorted(PRESETS)}") from exc


# ---- estimators/filters.py ----
"""Band-pass filtering and analytic-signal helpers shared by all estimators."""


import numpy as np
from scipy.signal import butter, filtfilt, hilbert


def bandpass(x: np.ndarray, fs: float, low: float, high: float, order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth band-pass along the last axis.

    Raises ``ValueError`` if the band is not strictly inside (0, fs/2), so a
    gamma band that exceeds Nyquist fails loudly instead of being clipped.
    """
    nyq = fs / 2.0
    if not (0.0 < low < high < nyq):
        raise ValueError(
            f"band ({low}, {high}) Hz must lie strictly inside (0, {nyq}) Hz for fs={fs} Hz"
        )
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, np.asarray(x, dtype=float), axis=-1)


def analytic_phase(x: np.ndarray) -> np.ndarray:
    return np.angle(hilbert(x, axis=-1))


def analytic_amplitude(x: np.ndarray) -> np.ndarray:
    return np.abs(hilbert(x, axis=-1))


def zscore_channels(x: np.ndarray) -> np.ndarray:
    """Per-channel z-score of an array of shape (channels, time)."""
    x = np.asarray(x, dtype=float)
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True) + 1e-12
    return (x - mu) / sd


# ---- estimators/dfa.py ----
"""Detrended fluctuation analysis with an explicit short-series policy.

The released notebooks contain two DFA implementations with different scale
grids (``min_win=16`` versus ``min_box=8``) and different behaviour on short
series. This module replaces both. The scale grid is always ascending, the
series length is checked against the configuration, and the result carries
the scales actually used so that an audit can state them.
"""


import warnings
from dataclasses import dataclass, field

import numpy as np



class ShortSeriesWarning(UserWarning):
    """Raised when a DFA exponent is estimated from fewer samples than advised."""


@dataclass
class DFAResult:
    alpha: float
    scales: np.ndarray = field(repr=False)
    fluctuation: np.ndarray = field(repr=False)
    n_samples: int = 0
    short_series: bool = False
    r_squared: float = float("nan")


def dfa_exponent(x: np.ndarray, config: DFAConfig = DFAConfig()) -> DFAResult:
    """DFA scaling exponent alpha of a one-dimensional signal.

    alpha ~ 0.5 for white noise, ~1.0 for 1/f noise and ~1.5 for Brownian
    motion. The input is demeaned and integrated before windowed detrending
    with a polynomial of ``config.detrend_order``.

    Series shorter than ``config.hard_min_samples`` return NaN. Series
    between ``hard_min_samples`` and ``min_samples`` return an exponent with
    ``short_series=True`` and a ``ShortSeriesWarning``.
    """
    x = np.asarray(x, dtype=float).ravel()
    n = x.size
    if n < config.hard_min_samples:
        return DFAResult(float("nan"), np.array([]), np.array([]), n, True)
    short = n < config.min_samples
    if short:
        warnings.warn(
            f"DFA on {n} samples is below the advised minimum of {config.min_samples}; "
            "treat the exponent as directional only",
            ShortSeriesWarning,
            stacklevel=2,
        )

    max_window = max(int(n * config.max_window_fraction), config.min_window + 1)
    if max_window <= config.min_window:
        return DFAResult(float("nan"), np.array([]), np.array([]), n, True)
    scales = np.unique(
        np.logspace(np.log10(config.min_window), np.log10(max_window), config.n_scales).astype(int)
    )
    scales = scales[scales >= 4]

    y = np.cumsum(x - x.mean())
    fluct = []
    used = []
    for s in scales:
        n_seg = n // s
        if n_seg < 2:
            continue
        segs = y[: n_seg * s].reshape(n_seg, s)
        t = np.arange(s, dtype=float)
        # Vectorised polynomial detrending of every segment at once.
        V = np.vander(t, config.detrend_order + 1)
        coef, *_ = np.linalg.lstsq(V, segs.T, rcond=None)
        resid = segs - (V @ coef).T
        rms = np.sqrt(np.mean(resid**2, axis=1))
        fluct.append(rms.mean())
        used.append(s)
    if len(used) < 2:
        return DFAResult(float("nan"), np.array(used), np.array(fluct), n, True)

    log_s = np.log(np.asarray(used, dtype=float))
    log_f = np.log(np.asarray(fluct, dtype=float))
    slope, intercept = np.polyfit(log_s, log_f, 1)
    pred = slope * log_s + intercept
    ss_res = float(np.sum((log_f - pred) ** 2))
    ss_tot = float(np.sum((log_f - log_f.mean()) ** 2)) + 1e-18
    return DFAResult(
        alpha=float(slope),
        scales=np.asarray(used),
        fluctuation=np.asarray(fluct),
        n_samples=n,
        short_series=short,
        r_squared=1.0 - ss_res / ss_tot,
    )


def dfa_channels(X: np.ndarray, config: DFAConfig = DFAConfig()) -> np.ndarray:
    """DFA exponent of every row of an array of shape (channels, time)."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    return np.array([dfa_exponent(row, config).alpha for row in X])


# ---- estimators/pac.py ----
"""Theta-gamma phase-amplitude coupling: Tort modulation index and surrogates."""


from dataclasses import dataclass

import numpy as np



def tort_modulation_index(
    phase: np.ndarray, amplitude: np.ndarray, n_bins: int = 36, amplitude_guard: float = 1e-3
) -> float:
    """Kullback-Leibler modulation index of Tort et al. (2010).

    The amplitude-by-phase profile over ``n_bins`` bins is normalised to a
    distribution and its divergence from uniform is returned. The amplitude
    guard returns 0 when the mean envelope is below ``amplitude_guard``,
    which is the paper's protection against spurious coupling from a
    near-empty gamma band (the seizure case).
    """
    phase = np.asarray(phase, dtype=float)
    amplitude = np.asarray(amplitude, dtype=float)
    if float(amplitude.mean()) < amplitude_guard:
        return 0.0
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    idx = np.clip(np.digitize(phase, edges) - 1, 0, n_bins - 1)
    sums = np.bincount(idx, weights=amplitude, minlength=n_bins)
    counts = np.bincount(idx, minlength=n_bins)
    profile = np.where(counts > 0, sums / np.maximum(counts, 1), 0.0)
    total = profile.sum()
    if total < 1e-12:
        return 0.0
    p = profile / total
    q = 1.0 / n_bins
    return float(np.sum(p * np.log(p / q + 1e-12)))


@dataclass
class PACResult:
    modulation_index: float
    per_channel: np.ndarray
    surrogate_mean: float = float("nan")
    surrogate_sd: float = float("nan")
    surrogate_z: float = float("nan")


def theta_gamma_pac(
    X: np.ndarray,
    fs: float,
    config: PACConfig = PACConfig(),
    surrogates: bool = False,
    rng: np.random.Generator | None = None,
) -> PACResult:
    """Channel-averaged theta-gamma modulation index of an array (channels, time).

    With ``surrogates=True`` a time-shift surrogate distribution is built by
    rolling the gamma envelope by a random lag in [T/4, 3T/4) relative to the
    theta phase, and a z-score is reported. The z-score is a diagnostic and
    never enters the composite, matching the paper.

    Time-shift surrogates only destroy coupling when the theta phase is not
    strictly periodic over the shift: for a stationary sinusoidal theta a
    shift merely rotates the preferred phase, to which the Tort index is
    invariant, so the surrogate z-score is uninformative on such signals.
    """
    X = np.atleast_2d(np.asarray(X, dtype=float))
    theta = bandpass(X, fs, *config.theta_band, order=config.filter_order)
    gamma = bandpass(X, fs, *config.gamma_band, order=config.filter_order)
    phase = analytic_phase(theta)
    amp = analytic_amplitude(gamma)
    per_channel = np.array(
        [tort_modulation_index(phase[c], amp[c], config.n_bins, config.amplitude_guard) for c in range(X.shape[0])]
    )
    result = PACResult(float(per_channel.mean()), per_channel)
    if not surrogates:
        return result
    rng = np.random.default_rng() if rng is None else rng
    T = X.shape[1]
    surr = np.empty(config.n_surrogates)
    for k in range(config.n_surrogates):
        shift = int(rng.integers(T // 4, 3 * T // 4))
        amp_s = np.roll(amp, shift, axis=-1)
        surr[k] = np.mean(
            [tort_modulation_index(phase[c], amp_s[c], config.n_bins, config.amplitude_guard) for c in range(X.shape[0])]
        )
    result.surrogate_mean = float(surr.mean())
    result.surrogate_sd = float(surr.std(ddof=1)) if config.n_surrogates > 1 else float("nan")
    result.surrogate_z = (
        float((result.modulation_index - result.surrogate_mean) / (result.surrogate_sd + 1e-12))
    )
    return result


# ---- estimators/lzc.py ----
"""Lempel-Ziv complexity of binarised envelopes with explicit normalisation.

The paper normalises the LZ76 phrase count by n / log2(n), which brings a
random binary sequence close to one. The core notebook's ``lempel_ziv_complexity``
returns the raw count divided by n, which is roughly log2(n) times smaller
and makes the complexity term inside D almost inert. Both normalisations are
available here under their own names so that the difference is visible.
"""


import numpy as np



def lz76_phrase_count(binary: np.ndarray) -> int:
    """Number of phrases in the Lempel-Ziv 1976 parsing of a binary sequence.

    Implements the Kaspar and Schuster (1987) algorithm, in which each new
    phrase is the shortest substring not reproducible from the prefix that
    precedes it (the match may extend into the phrase itself).
    """
    s = np.asarray(binary).astype(bool).astype(np.uint8).tobytes()
    n = len(s)
    if n == 0:
        return 0
    # Each new phrase is the shortest substring starting at i that does not occur
    # in s[0 : i + l - 1], i.e. the match may overlap the phrase itself except
    # its last symbol. The substring search runs in C, so the parse of a few
    # thousand symbols takes milliseconds.
    i, c = 0, 0
    while i < n:
        l = 1
        while i + l <= n and s[i : i + l] in s[: i + l - 1]:
            l += 1
        c += 1
        i += l
    return c


def normalised_lzc(binary: np.ndarray, normalisation: str = "n_over_log2n") -> float:
    binary = np.asarray(binary)
    n = binary.size
    if n < 2:
        return 0.0
    c = lz76_phrase_count(binary)
    if normalisation == "n_over_log2n":
        return float(c / (n / np.log2(n)))
    if normalisation == "c_over_n":
        return float(c / n)
    raise ValueError(f"unknown LZ normalisation {normalisation!r}")


def broadband_envelope_lzc(X: np.ndarray, fs: float, config: LZCConfig = LZCConfig()) -> np.ndarray:
    """Per-channel LZ complexity of the median-binarised broadband envelope."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    env = analytic_amplitude(bandpass(X, fs, *config.band, order=config.filter_order))
    out = []
    for row in env:
        binary = row > np.median(row)
        out.append(normalised_lzc(binary, config.normalisation))
    return np.asarray(out)


# ---- estimators/kuramoto.py ----
"""Kuramoto order parameter and metastability."""


import numpy as np



def order_parameter(phases: np.ndarray) -> np.ndarray:
    """|mean_i exp(i theta_i(t))| for phases of shape (channels, time)."""
    phases = np.atleast_2d(np.asarray(phases, dtype=float))
    return np.abs(np.mean(np.exp(1j * phases), axis=0))


def metastability_from_phases(phases: np.ndarray) -> float:
    """Temporal standard deviation of the Kuramoto order parameter."""
    return float(np.std(order_parameter(phases)))


def band_metastability(X: np.ndarray, fs: float, config: MetastabilityConfig = MetastabilityConfig()) -> tuple[float, np.ndarray]:
    """Metastability of the band-limited analytic phase of (channels, time).

    Returns ``(M, R_t)``. With two channels the order parameter reduces to a
    pairwise phase-lag measure; the caller is expected to flag this.
    """
    X = np.atleast_2d(np.asarray(X, dtype=float))
    phases = analytic_phase(bandpass(X, fs, *config.band, order=config.filter_order))
    R_t = order_parameter(phases)
    return float(np.std(R_t)), R_t


# ---- estimators/spectral.py ----
"""Single-metric baselines used for benchmarking: spectral slope, alpha power,
effective rank, autocorrelation time and trajectory velocity."""


import numpy as np
from scipy.signal import welch


def spectral_slope(X: np.ndarray, fs: float, fmin: float = 2.0, fmax: float = 45.0) -> float:
    """Aperiodic log-log slope of the Welch spectrum between fmin and fmax, averaged over channels."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    slopes = []
    for row in X:
        f, psd = welch(row, fs=fs, nperseg=min(512, row.size // 2))
        mask = (f >= fmin) & (f <= fmax)
        if mask.sum() < 3:
            continue
        slopes.append(np.polyfit(np.log10(f[mask]), np.log10(psd[mask] + 1e-12), 1)[0])
    return float(np.mean(slopes)) if slopes else float("nan")


def band_power_fraction(X: np.ndarray, fs: float, band: tuple[float, float] = (8.0, 13.0), total: tuple[float, float] = (1.0, 45.0)) -> float:
    """Relative power in ``band`` as a fraction of power in ``total``, averaged over channels."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    fracs = []
    for row in X:
        f, psd = welch(row, fs=fs, nperseg=min(512, row.size // 2))
        num = psd[(f >= band[0]) & (f <= band[1])].sum()
        den = psd[(f >= total[0]) & (f <= total[1])].sum() + 1e-18
        fracs.append(num / den)
    return float(np.mean(fracs))


def effective_rank(H: np.ndarray) -> float:
    """Roy and Vetterli (2007) effective rank of a matrix (time, features)."""
    H = np.asarray(H, dtype=float)
    H = H - H.mean(axis=0, keepdims=True)
    s = np.linalg.svd(H, compute_uv=False)
    p = s / (s.sum() + 1e-18)
    p = p[p > 0]
    return float(np.exp(-np.sum(p * np.log(p))))


def autocorrelation_time(x: np.ndarray, max_lag: int | None = None) -> float:
    """Integrated autocorrelation time: 1 + 2 * sum of positive autocorrelations."""
    x = np.asarray(x, dtype=float).ravel()
    x = x - x.mean()
    n = x.size
    if n < 4:
        return float("nan")
    max_lag = n // 4 if max_lag is None else max_lag
    var = np.dot(x, x) / n + 1e-18
    tau = 1.0
    for lag in range(1, max_lag):
        r = np.dot(x[:-lag], x[lag:]) / (n - lag) / var
        if r <= 0:
            break
        tau += 2.0 * r
    return float(tau)


def trajectory_velocity(H: np.ndarray) -> float:
    """Mean Euclidean step length between consecutive rows of (time, features)."""
    H = np.asarray(H, dtype=float)
    if H.shape[0] < 2:
        return float("nan")
    return float(np.mean(np.linalg.norm(np.diff(H, axis=0), axis=1)))


# ---- components.py ----
"""The three Psi components computed from a multichannel signal.

``compute_components`` returns the raw quantities (channel-averaged DFA
exponent, Tort modulation index, broadband LZ complexity, D, M) together with
the configuration used. H_eff is a mapping applied afterwards, because for
the triangular and per-subject modes it depends on an ensemble (the alpha
range or a subject's reference condition), not on a single signal.
"""


from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde



@dataclass
class Components:
    alpha_dfa: float
    alpha_per_channel: np.ndarray
    i_phi_a: float
    lz: float
    d: float
    m: float
    n_channels: int
    n_samples: int
    fs: float
    config_name: str
    pac_surrogate_z: float = float("nan")
    short_series: bool = False
    two_channel_metastability: bool = False

    def to_row(self) -> dict:
        row = asdict(self)
        row.pop("alpha_per_channel")
        return row


def compute_components(
    X: np.ndarray,
    config: AuditConfig,
    zscore: bool = True,
    pac_surrogates: bool = False,
    rng: np.random.Generator | None = None,
) -> Components:
    """Raw components of an array of shape (channels, time) at ``config.fs``."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    if zscore:
        X = zscore_channels(X)
    C, T = X.shape

    alphas = dfa_channels(X, config.dfa)
    alpha = float(np.nanmean(alphas))

    pac = theta_gamma_pac(X, config.fs, config.pac, surrogates=pac_surrogates, rng=rng)
    lz = float(np.mean(broadband_envelope_lzc(X, config.fs, config.lzc)))
    d = pac.modulation_index * (1.0 + config.composite.lam * lz)

    m, _ = band_metastability(X, config.fs, config.metastability)

    return Components(
        alpha_dfa=alpha,
        alpha_per_channel=alphas,
        i_phi_a=pac.modulation_index,
        lz=lz,
        d=float(d),
        m=float(m),
        n_channels=C,
        n_samples=T,
        fs=config.fs,
        config_name=config.name,
        pac_surrogate_z=pac.surrogate_z,
        short_series=T < config.dfa.min_samples,
        two_channel_metastability=C <= 2,
    )


# ---------------------------------------------------------------------------
# H_eff mappings
# ---------------------------------------------------------------------------


def heff_gaussian(alpha: np.ndarray | float, h_opt: float, sigma_h: float) -> np.ndarray | float:
    """Paper Eq. 3: exp(-(alpha - H_opt)^2 / (2 sigma_H^2))."""
    a = np.asarray(alpha, dtype=float)
    return np.exp(-((a - h_opt) ** 2) / (2.0 * sigma_h**2))


def heff_triangular(alpha: np.ndarray | float, h_opt: float, alpha_range: float) -> np.ndarray | float:
    """Paper Eq. 4: max(0, 1 - |alpha - H_opt| / alpha_range)."""
    a = np.asarray(alpha, dtype=float)
    if alpha_range <= 0:
        return np.zeros_like(a)
    return np.maximum(0.0, 1.0 - np.abs(a - h_opt) / alpha_range)


def _kde_mode(values: np.ndarray, grid_n: int = 512) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return float(np.median(values)) if values.size else float("nan")
    try:
        kde = gaussian_kde(values, bw_method="scott")
        lo, hi = values.min(), values.max()
        pad = (hi - lo) * 0.05 if hi > lo else 0.05
        grid = np.linspace(lo - pad, hi + pad, grid_n)
        return float(grid[np.argmax(kde(grid))])
    except Exception:
        return float(np.median(values))


def heff_ensemble(alpha: np.ndarray, config: HeffConfig) -> np.ndarray:
    """H_eff for an ensemble of DFA exponents under a Gaussian or triangular mapping.

    For the triangular mode the denominator is ``config.alpha_range`` when
    given, otherwise the empirical range of the ensemble (the released
    notebook's behaviour).
    """
    alpha = np.asarray(alpha, dtype=float)
    if config.mode == "gaussian":
        return heff_gaussian(alpha, config.h_opt, config.sigma_h)
    if config.mode == "triangular":
        rng_ = config.alpha_range
        if rng_ is None:
            rng_ = float(np.nanmax(alpha) - np.nanmin(alpha))
        return heff_triangular(alpha, config.h_opt, rng_)
    raise ValueError("per_subject mode requires heff_per_subject with subject and stage labels")


def heff_per_subject(
    df: pd.DataFrame,
    config: HeffConfig,
    alpha_col: str = "alpha_dfa",
    subject_col: str = "subject",
    stage_col: str = "stage",
    reference_stage: str = "W",
) -> tuple[np.ndarray, dict]:
    """Per-subject Wake-anchored triangular H_eff (paper Section 4.2, real EEG).

    The anchor is the per-subject median (paper) or KDE mode (released
    notebook) of the reference-stage exponents, and the scale is the
    per-subject range of exponents across all retained epochs, with a fall
    back to the global range when the subject range is below
    ``config.min_subject_range``. Returns the H_eff array aligned with ``df``
    and a dictionary of per-subject anchors.
    """
    heff = np.full(len(df), np.nan)
    anchors: dict = {}
    global_range = float(df[alpha_col].max() - df[alpha_col].min())
    global_ref_median = float(df.loc[df[stage_col] == reference_stage, alpha_col].median())
    for subj, sub in df.groupby(subject_col):
        ref = sub.loc[sub[stage_col] == reference_stage, alpha_col].to_numpy()
        if ref.size >= 3:
            anchor = _kde_mode(ref) if config.anchor == "kde_mode" else float(np.median(ref))
        else:
            anchor = global_ref_median
        subj_range = float(sub[alpha_col].max() - sub[alpha_col].min())
        if subj_range < config.min_subject_range:
            subj_range = global_range
        anchors[subj] = {"anchor": anchor, "range": subj_range, "n_reference": int(ref.size)}
        heff[df.index.get_indexer(sub.index)] = heff_triangular(sub[alpha_col].to_numpy(), anchor, subj_range)
    return heff, anchors


# ---- composite.py ----
"""The composite index Psi with a first-class normalisation reference.

Psi values are only comparable when they share the same normalisation
reference and the same H_eff mapping. ``NormalisationReference`` records the
min and max of H_eff, D and M on a named reference ensemble so that a report
can state what any Psi was normalised against, and so that synthetic and
empirical values are never silently compared.
"""


import json
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd



@dataclass
class NormalisationReference:
    name: str
    bounds: dict[str, tuple[float, float]] = field(default_factory=dict)
    n_rows: int = 0
    heff_mode: str = ""
    config_name: str = ""

    @classmethod
    def fit(
        cls,
        df: pd.DataFrame,
        name: str,
        cols: tuple[str, str, str] = ("heff", "d", "m"),
        heff_mode: str = "",
        config_name: str = "",
    ) -> "NormalisationReference":
        bounds = {c: (float(np.nanmin(df[c])), float(np.nanmax(df[c]))) for c in cols}
        return cls(name=name, bounds=bounds, n_rows=len(df), heff_mode=heff_mode, config_name=config_name)

    def normalise(self, values: np.ndarray | pd.Series, col: str, clip: bool = True) -> np.ndarray:
        lo, hi = self.bounds[col]
        v = (np.asarray(values, dtype=float) - lo) / (hi - lo + 1e-12)
        return np.clip(v, 0.0, 1.0) if clip else v

    def to_json(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2))

    @classmethod
    def from_json(cls, path: str | Path) -> "NormalisationReference":
        d = json.loads(Path(path).read_text())
        d["bounds"] = {k: tuple(v) for k, v in d["bounds"].items()}
        return cls(**d)


def psi(
    heff: np.ndarray | pd.Series,
    d: np.ndarray | pd.Series,
    m: np.ndarray | pd.Series,
    reference: NormalisationReference,
    weights: CompositeConfig = CompositeConfig(),
    clip: bool = True,
) -> np.ndarray:
    """Psi = w_H H_eff,norm + w_D D_norm + w_M M_norm against ``reference``."""
    return (
        weights.w_h * reference.normalise(heff, "heff", clip)
        + weights.w_d * reference.normalise(d, "d", clip)
        + weights.w_m * reference.normalise(m, "m", clip)
    )


def add_psi(
    df: pd.DataFrame,
    reference: NormalisationReference,
    weights: CompositeConfig = CompositeConfig(),
    clip: bool = True,
) -> pd.DataFrame:
    """Return a copy of ``df`` with normalised columns and ``psi`` added."""
    out = df.copy()
    for c in ("heff", "d", "m"):
        out[f"{c}_norm"] = reference.normalise(out[c], c, clip)
    out["psi"] = weights.w_h * out["heff_norm"] + weights.w_d * out["d_norm"] + weights.w_m * out["m_norm"]
    out.attrs["normalisation_reference"] = reference.name
    return out


def fractional_contributions(row: pd.Series, weights: CompositeConfig = CompositeConfig()) -> dict[str, float]:
    """Share of Psi carried by each component for one normalised row (paper Section 6.9)."""
    parts = {
        "heff": weights.w_h * float(row["heff_norm"]),
        "d": weights.w_d * float(row["d_norm"]),
        "m": weights.w_m * float(row["m_norm"]),
    }
    total = sum(parts.values()) + 1e-12
    return {k: v / total for k, v in parts.items()}


# ---- simulate.py ----
"""Generative model of nine EEG-like brain states (paper Section 5).

This is a faithful port of ``generate_consciousness_state_signal`` and
``_sim_rem`` from the released state-simulator notebook, with two changes.
All randomness goes through an explicit ``numpy.random.Generator`` so that a
realisation is reproducible from a seed, and the state table is exposed as
data so that the parameters can be reported.

Two facts about the released generator are worth knowing. ``dreaming`` is an
alias of ``psychedelic`` in the notebook (the paper's Table 2 lists separate
Dreaming parameters, K = 0.42 and eta_PAC = 0.20, that do not correspond to
any code path used in the Monte Carlo), and the separate ``rem_sleep``
generator was not part of the nine-state ensemble. Both behaviours are
preserved here and flagged in ``STATE_NOTES``.
"""


import numpy as np


BAND_EDGES = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 80.0),
}

STATE_PARAMS: dict[str, dict] = {
    "wake": {
        "bands": {"delta": 0.3, "theta": 0.4, "alpha": 0.8, "beta": 0.6, "gamma": 0.5},
        "coupling": 0.6, "local_sync": 0.3, "global_sync": 0.5, "pac_strength": 0.4,
    },
    "task_engaged": {
        "bands": {"delta": 0.2, "theta": 0.5, "alpha": 0.7, "beta": 0.7, "gamma": 0.6},
        "coupling": 0.55, "local_sync": 0.50, "global_sync": 0.35, "pac_strength": 0.55,
    },
    "sleep": {
        "bands": {"delta": 0.95, "theta": 0.7, "alpha": 0.2, "beta": 0.1, "gamma": 0.05},
        "coupling": 0.4, "local_sync": 0.7, "global_sync": 0.3, "pac_strength": 0.05,
    },
    "anesthesia": {
        "bands": {"delta": 0.9, "theta": 0.5, "alpha": 0.5, "beta": 0.05, "gamma": 0.02},
        "coupling": 0.10, "local_sync": 0.82, "global_sync": 0.08, "pac_strength": 0.005,
    },
    "psychedelic": {
        "bands": {"delta": 0.2, "theta": 0.4, "alpha": 0.6, "beta": 0.8, "gamma": 1.0},
        "coupling": 0.7, "local_sync": 0.2, "global_sync": 0.4, "pac_strength": 0.7,
    },
    "seizure": {
        "bands": {"delta": 0.0, "theta": 0.0, "alpha": 0.0, "beta": 0.0, "gamma": 0.0},
        "coupling": 0.0, "local_sync": 0.0, "global_sync": 1.0, "pac_strength": 0.0,
    },
    "minimally_conscious": {
        "bands": {"delta": 0.55, "theta": 0.35, "alpha": 0.25, "beta": 0.05, "gamma": 0.0},
        "coupling": 0.32, "local_sync": 0.45, "global_sync": 0.12, "pac_strength": 0.060,
    },
    "non_conscious": {
        "bands": {"delta": 0.0, "theta": 0.0, "alpha": 0.0, "beta": 0.0, "gamma": 0.0},
        "coupling": 0.0, "local_sync": 0.0, "global_sync": 0.0, "pac_strength": 0.0,
    },
}

STATE_ALIASES = {"dreaming": "psychedelic", "anaesthesia": "anesthesia"}

STATE_NOTES = {
    "dreaming": "alias of 'psychedelic' in the released generator; paper Table 2 lists distinct parameters",
    "rem_sleep": "separate generator, not part of the nine-state Monte Carlo ensemble",
    "seizure": "frequency-domain 1/f^2 burst in 0.5-4 Hz tiled across channels; D = 0 by the amplitude guard",
    "non_conscious": "frequency-domain 1/f^2 in 0.3-5.5 Hz tiled across channels",
}

#: The nine states of the paper's Monte Carlo ensemble, in the paper's order.
NINE_STATES = [
    "psychedelic", "wake", "task_engaged", "dreaming", "sleep",
    "minimally_conscious", "anesthesia", "non_conscious", "seizure",
]

STATE_DISPLAY = {
    "psychedelic": "Psychedelic",
    "wake": "Wake",
    "task_engaged": "Task-engaged",
    "dreaming": "Dreaming",
    "sleep": "Sleep (NREM-like)",
    "minimally_conscious": "Minimally Conscious",
    "anesthesia": "Anaesthesia",
    "non_conscious": "Non-conscious",
    "seizure": "Seizure",
    "rem_sleep": "REM sleep",
}


def fractional_gaussian_noise(hurst: float, n_samples: int, rng: np.random.Generator | int | None = None,
                              n_channels: int = 1) -> np.ndarray:
    """Exact fractional Gaussian noise by the Davies-Harte circulant embedding, shape (n_channels, n_samples).

    DFA exponent alpha = H for this process, which makes it the reference for a scale-free
    signal whose generating mechanism is known.
    """
    rng = np.random.default_rng(rng) if not isinstance(rng, np.random.Generator) else rng
    n = int(n_samples)
    k = np.arange(n + 1, dtype=float)
    g = 0.5 * (np.abs(k - 1) ** (2 * hurst) - 2 * np.abs(k) ** (2 * hurst) + np.abs(k + 1) ** (2 * hurst))
    c = np.concatenate([g, g[-2:0:-1]])
    lam = np.fft.fft(c).real
    lam[lam < 0] = 0.0
    out = np.empty((n_channels, n))
    for ch in range(n_channels):
        z = rng.standard_normal(c.size) + 1j * rng.standard_normal(c.size)
        out[ch] = np.fft.fft(np.sqrt(lam / c.size) * z)[:n].real
    return out


def pink_noise(rng: np.random.Generator, n_channels: int, n_samples: int) -> np.ndarray:
    """Integrated white noise, z-scored per channel (the notebook's 1/f background)."""
    pink = np.cumsum(rng.standard_normal((n_channels, n_samples)), axis=-1)
    pink -= pink.mean(axis=-1, keepdims=True)
    pink /= pink.std(axis=-1, keepdims=True) + 1e-9
    return pink


def _tiled_power_law(rng: np.random.Generator, n_channels: int, n_samples: int, fs: float,
                     band: tuple[float, float], white_level: float) -> np.ndarray:
    f = np.fft.rfftfreq(n_samples, 1.0 / fs)
    f[0] = 1e-9
    brown = np.where((f >= band[0]) & (f <= band[1]), 1.0 / f**2, 0.0)
    white = np.where(f >= band[1], white_level, 0.0)
    spec = (brown + white) * (rng.standard_normal(f.size) + 1j * rng.standard_normal(f.size))
    x = np.fft.irfft(spec, n=n_samples)
    x -= x.mean()
    x /= x.std() + 1e-9
    return np.tile(x, (n_channels, 1))


def _network_coupling(rng: np.random.Generator, signal: np.ndarray, coupling: float,
                      local_sync: float) -> np.ndarray:
    n_channels, n_samples = signal.shape
    p_rewire = 0.05 if local_sync > 0.5 else 0.3
    G = nx.watts_strogatz_graph(n_channels, k=min(4, n_channels - 1), p=p_rewire, seed=int(rng.integers(0, 2**31 - 1)))
    A = nx.to_numpy_array(G)
    A = A / (A.sum(axis=1, keepdims=True) + 1e-9)
    amp = analytic_amplitude(signal)
    phase = analytic_phase(signal)
    for t in range(1, n_samples):
        prev = phase[:, t - 1]
        phase_diff = prev[:, None] - prev[None, :]
        phase[:, t] += coupling * 0.02 * (A * np.sin(phase_diff)).sum(axis=1)
    return amp * np.cos(phase)


def simulate_state(
    state: str,
    n_channels: int = 16,
    n_samples: int = 3000,
    fs: float = 250.0,
    rng: np.random.Generator | int | None = None,
) -> np.ndarray:
    """One realisation of a simulated state, shape (n_channels, n_samples), z-scored per channel."""
    rng = np.random.default_rng(rng) if not isinstance(rng, np.random.Generator) else rng
    state = STATE_ALIASES.get(state, state)
    if state == "rem_sleep":
        return simulate_rem(n_channels, n_samples, fs, rng)
    if state not in STATE_PARAMS:
        raise ValueError(f"unknown state {state!r}; choose from {sorted(STATE_PARAMS) + ['dreaming', 'rem_sleep']}")
    p = STATE_PARAMS[state]

    base = pink_noise(rng, n_channels, n_samples)
    signal = np.zeros_like(base)

    for band, weight in p["bands"].items():
        if weight <= 0:
            continue
        low, high = BAND_EDGES[band]
        g = bandpass(rng.standard_normal((1, n_samples)), fs, low, high)[0]
        g = (g - g.mean()) / (g.std() + 1e-9)
        local = bandpass(base, fs, low, high)
        local = (local - local.mean(axis=-1, keepdims=True)) / (local.std(axis=-1, keepdims=True) + 1e-9)
        signal += weight * ((1.0 - p["global_sync"]) * local + p["global_sync"] * g[None, :])

    if p["pac_strength"] > 0:
        theta_phase = analytic_phase(bandpass(signal, fs, *BAND_EDGES["theta"]))
        gamma = bandpass(rng.standard_normal((n_channels, n_samples)) * 0.5, fs, *BAND_EDGES["gamma"])
        pac_mod = (1.0 + p["pac_strength"] * np.sin(theta_phase)) / 2.0
        signal += pac_mod * gamma * 3.0

    if state == "seizure":
        signal = _tiled_power_law(rng, n_channels, n_samples, fs, (0.5, 4.0), 0.03)
    elif state == "non_conscious":
        signal = _tiled_power_law(rng, n_channels, n_samples, fs, (0.3, 5.5), 0.02)
    elif n_channels > 1 and p["coupling"] > 0:
        signal = _network_coupling(rng, signal, p["coupling"], p["local_sync"])

    signal -= signal.mean(axis=-1, keepdims=True)
    signal /= signal.std(axis=-1, keepdims=True) + 1e-9
    return signal


def simulate_rem(n_channels: int = 16, n_samples: int = 3000, fs: float = 250.0,
                 rng: np.random.Generator | int | None = None) -> np.ndarray:
    """The notebook's separate REM-sleep generator (not in the nine-state ensemble)."""
    rng = np.random.default_rng(rng) if not isinstance(rng, np.random.Generator) else rng
    bands = {"delta": 0.55, "theta": 0.60, "alpha": 0.40, "beta": 0.35, "gamma": 0.28}
    X = pink_noise(rng, n_channels, n_samples) * 0.2
    t = np.arange(n_samples) / fs
    for band, (lo, hi) in BAND_EDGES.items():
        fc = (lo + hi) / 2.0
        w = bands[band]
        gosc = np.sin(2 * np.pi * fc * t + rng.uniform(0, 2 * np.pi))
        for ch in range(n_channels):
            X[ch] += w * (0.6 * gosc + 0.4 * np.sin(2 * np.pi * fc * t + rng.uniform(0, 2 * np.pi)))
    th_ph = analytic_phase(bandpass(X, fs, 4.0, 8.0))
    for ch in range(n_channels):
        gf = bandpass((rng.standard_normal(n_samples) * 0.3)[None, :], fs, 30.0, 80.0)[0]
        X[ch] += (1 + 0.20 * np.sin(th_ph[ch])) / 2 * gf * 2.0
    X += 0.30 * np.sin(2 * np.pi * 0.5 * t)[None, :]
    X -= X.mean(axis=-1, keepdims=True)
    X /= X.std(axis=-1, keepdims=True) + 1e-9
    return X


def simulate_ensemble(
    states: list[str] = NINE_STATES,
    n_runs: int = 30,
    n_channels: int = 16,
    n_samples: int = 3000,
    fs: float = 250.0,
    seed: int = 0,
):
    """Yield ``(state, run_index, X)`` for a Monte Carlo ensemble with per-run seeds."""
    import zlib

    for state in states:
        state_code = zlib.crc32(state.encode()) % 10_007
        for run in range(n_runs):
            run_seed = (seed * 1_000_003 + state_code * 101 + run) % (2**32 - 1)
            yield state, run, simulate_state(state, n_channels, n_samples, fs, np.random.default_rng(run_seed))


# ---- validation.py ----
"""Validation of a scaling statistic before a regime label is assigned.

A fitted DFA exponent is a number whatever the signal. Before it is read
as evidence of scale-free organisation the fluctuation curve has to be a
straight line in log-log coordinates over the scale range, the spectrum
must not be dominated by a single peak, and the activity must be
stationary over the window. ``scaling_validity`` computes those checks and
returns them with a verdict. ``persistence`` characterises activity under
zero external drive (variance, stationarity, transient decay) so that
autonomous persistence can be reported separately from the scaling claim.
"""


from dataclasses import asdict, dataclass

import numpy as np
from scipy.signal import welch



@dataclass
class ScalingValidity:
    alpha: float
    r_squared: float
    slope_short: float
    slope_long: float
    crossover: float
    max_local_deviation: float
    spectral_concentration: float
    spectral_flatness: float
    peak_frequency: float
    surrogate_alpha_mean: float
    surrogate_alpha_sd: float
    stationarity_ratio: float
    n_scales: int
    scale_decades: float
    valid: bool
    reasons: list[str]
    flags: list[str] = None

    def to_row(self) -> dict:
        d = asdict(self)
        d["reasons"] = "; ".join(self.reasons)
        d["flags"] = ",".join(self.flags or [])
        return d


def _phase_randomised(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    X = np.fft.rfft(x - x.mean())
    phases = rng.uniform(0, 2 * np.pi, X.size)
    phases[0] = 0.0
    if x.size % 2 == 0:
        phases[-1] = 0.0
    return np.fft.irfft(np.abs(X) * np.exp(1j * phases), n=x.size)


def scaling_validity(
    x: np.ndarray,
    fs: float = 1.0,
    config: DFAConfig = DFAConfig(min_window=16, max_window_fraction=0.1, n_scales=12),
    n_surrogates: int = 20,
    rng: np.random.Generator | None = None,
    r2_min: float = 0.98,
    local_dev_max: float = 0.35,
    concentration_max: float = 0.5,
    stationarity_max: float = 3.0,
    min_decades: float = 1.0,
) -> ScalingValidity:
    """Checks that a one-dimensional series supports a scale-free reading of its DFA exponent.

    The checks are: log-log linearity of the fluctuation function (R^2 and the
    largest deviation of a local two-point slope from the global slope), the
    short-scale versus long-scale slope ratio, spectral concentration (fraction
    of non-DC Welch power in the three largest bins) and spectral flatness,
    stationarity (ratio of the larger to the smaller half-window variance), and
    a phase-randomised surrogate check (DFA is a spectral quantity, so the
    exponent should be preserved under phase randomisation; a large difference
    flags a nonlinear or non-stationary structure that the exponent does not
    describe). The verdict is the conjunction of the thresholds.
    """
    rng = np.random.default_rng(0) if rng is None else rng
    x = np.asarray(x, dtype=float).ravel()
    res = dfa_exponent(x, config)
    reasons: list[str] = []
    if not np.isfinite(res.alpha) or res.scales.size < 4:
        nan = float("nan")
        return ScalingValidity(alpha=nan, r_squared=nan, slope_short=nan, slope_long=nan, crossover=nan,
                               max_local_deviation=nan, spectral_concentration=nan, spectral_flatness=nan,
                               peak_frequency=nan, surrogate_alpha_mean=nan, surrogate_alpha_sd=nan,
                               stationarity_ratio=nan, n_scales=int(res.scales.size), scale_decades=0.0,
                               valid=False, reasons=["too few scales or constant series"], flags=["range"])
    ls, lf = np.log(res.scales), np.log(res.fluctuation)
    k = max(2, len(ls) // 2)
    slope_short = float(np.polyfit(ls[:k], lf[:k], 1)[0])
    slope_long = float(np.polyfit(ls[-k:], lf[-k:], 1)[0])
    # local slopes over three consecutive scales, which damps the noise of the largest scales
    local = np.array([np.polyfit(ls[i:i + 3], lf[i:i + 3], 1)[0] for i in range(len(ls) - 2)])
    max_dev = float(np.max(np.abs(local - res.alpha)))
    decades = float((ls[-1] - ls[0]) / np.log(10))

    f, p = welch(x - x.mean(), fs=fs, nperseg=min(2048, max(64, x.size // 4)))
    p = p[1:]
    conc = float(np.sort(p)[-3:].sum() / (p.sum() + 1e-18))
    flat = float(np.exp(np.mean(np.log(p + 1e-18))) / (p.mean() + 1e-18))
    peak = float(f[1:][np.argmax(p)])

    h = x.size // 2
    v1, v2 = x[:h].var(), x[h:].var()
    stat_ratio = float(max(v1, v2) / (min(v1, v2) + 1e-18))

    surr = np.array([dfa_exponent(_phase_randomised(x, rng), config).alpha for _ in range(n_surrogates)])
    s_mean, s_sd = float(np.nanmean(surr)), float(np.nanstd(surr))

    flags: list[str] = []
    if res.r_squared < r2_min:
        reasons.append(f"log-log fit R2 {res.r_squared:.3f} < {r2_min}"); flags.append("linearity")
    if max_dev > local_dev_max:
        reasons.append(f"local slope deviates by {max_dev:.2f} (crossover)"); flags.append("crossover")
    if conc > concentration_max:
        reasons.append(f"spectral concentration {conc:.2f} (dominant peak at {peak:.2f})"); flags.append("peak")
    if stat_ratio > stationarity_max:
        reasons.append(f"non-stationary (variance ratio {stat_ratio:.1f})"); flags.append("nonstationary")
    if decades < min_decades:
        reasons.append(f"scale range {decades:.2f} decades"); flags.append("range")
    if abs(res.alpha - s_mean) > 3 * (s_sd + 0.02):
        reasons.append("exponent not preserved under phase randomisation"); flags.append("surrogate")
    return ScalingValidity(
        alpha=float(res.alpha), r_squared=float(res.r_squared), slope_short=slope_short, slope_long=slope_long,
        crossover=float(slope_short - slope_long), max_local_deviation=max_dev, spectral_concentration=conc,
        spectral_flatness=flat, peak_frequency=peak, surrogate_alpha_mean=s_mean, surrogate_alpha_sd=s_sd,
        stationarity_ratio=stat_ratio, n_scales=int(res.scales.size), scale_decades=decades,
        valid=not reasons, reasons=reasons, flags=flags,
    )


def channel_scaling_validity(X: np.ndarray, fs: float = 1.0, max_channels: int = 8, **kw) -> dict:
    """Validity summary over up to ``max_channels`` rows of (channels, time): mean fields and the fraction valid."""
    X = np.atleast_2d(np.asarray(X, dtype=float))
    live = [c for c in range(X.shape[0]) if X[c].std() > 1e-9][:max_channels]
    if not live:
        return {k: float("nan") for k in ("alpha", "r_squared", "slope_short", "slope_long", "crossover", "max_local_deviation",
                                           "spectral_concentration", "spectral_flatness", "stationarity_ratio")} | {"fraction_valid": 0.0, "flags": "range:all"}
    rows = [scaling_validity(X[c], fs, **kw) for c in live]
    out = {k: float(np.nanmean([getattr(r, k) for r in rows])) for k in
           ("alpha", "r_squared", "slope_short", "slope_long", "crossover", "max_local_deviation",
            "spectral_concentration", "spectral_flatness", "stationarity_ratio")}
    out["fraction_valid"] = float(np.mean([r.valid for r in rows]))
    counts = {}
    for r in rows:
        for f in (r.flags or []):
            counts[f] = counts.get(f, 0) + 1
    out["flags"] = ",".join(f"{k}:{v}/{len(rows)}" for k, v in sorted(counts.items()))
    return out


@dataclass
class Persistence:
    variance: float
    variance_first_half: float
    variance_second_half: float
    decay_slope: float
    persistent: bool
    stationary: bool

    def to_row(self) -> dict:
        return asdict(self)


def persistence(X: np.ndarray, n_windows: int = 8, variance_min: float = 1e-6, stationarity_max: float = 3.0) -> Persistence:
    """Activity under zero external drive: is it present, and is it sustained rather than a transient?

    ``decay_slope`` is the slope of log variance against window index; a
    strongly negative value is a decaying transient.
    """
    X = np.atleast_2d(np.asarray(X, dtype=float))
    T = X.shape[1]
    v = float(X.var(axis=1).mean())
    h = T // 2
    v1, v2 = float(X[:, :h].var(axis=1).mean()), float(X[:, h:].var(axis=1).mean())
    w = T // n_windows
    wins = np.array([X[:, i * w:(i + 1) * w].var(axis=1).mean() for i in range(n_windows)]) + 1e-18
    slope = float(np.polyfit(np.arange(n_windows), np.log(wins), 1)[0])
    present = v > variance_min
    stationary = present and max(v1, v2) / (min(v1, v2) + 1e-18) < stationarity_max and slope > -0.3
    return Persistence(v, v1, v2, slope, present and stationary, stationary)


# ---- systems.py ----
"""Reference systems with known dynamical regimes for validating the
attribution protocol. Each is a callable from an input array (units, time)
to an activity array (units, time); a zero input gives the autonomous
condition. The system boundary (what counts as internal, including any
internal noise source) is stated in each docstring.

The cases are:

* ``FeedforwardFilter``     transmits a structured input (no state of its own)
* ``LeakyNetwork``          stable linear system with its own response timescale;
                            decays to silence without input
* ``ChaoticRateNetwork``    large tanh rate network with one fixed matrix swept
                            across gain; autonomous and chaotic above g = 1 but
                            with exponentially decaying correlations, hence not
                            scale-free
* ``OscillatorNetwork``     coupled Stuart-Landau oscillators; autonomous limit
                            cycle, the deliberate negative for a scaling claim
* ``HeterogeneousTimescaleNetwork``
                            leaky units with log-uniformly distributed time
                            constants driven by internal noise; 1/f activity by
                            the superposition mechanism, scale-free within the
                            band of time constants for a known reason

``CriticalBranchingNetwork`` is a sixth case, a probabilistic branching
network whose correlation time diverges as the branching ratio approaches
one, which gives near-scale-free activity over a bounded range.
"""


from dataclasses import dataclass, field

import numpy as np



@dataclass
class FeedforwardFilter:
    """y = bandpass(tanh(W u)). Boundary: W and the filter. No internal noise, no state."""

    n_units: int
    fs: float
    band: tuple[float, float]
    seed: int = 0
    W: np.ndarray = field(init=False, repr=False)

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.W = rng.standard_normal((self.n_units, self.n_units)) / np.sqrt(self.n_units)

    def __call__(self, u: np.ndarray) -> np.ndarray:
        return bandpass(np.tanh(self.W @ u), self.fs, *self.band)


@dataclass
class LeakyNetwork:
    """x_{t+1} = a x_t + (1 - a) W u_t with 0 < a < 1. Boundary: a, W. Silent without input,
    but owns a relaxation timescale tau = -1 / ln(a) that shapes any input it receives."""

    n_units: int
    tau: float = 20.0
    seed: int = 0
    W: np.ndarray = field(init=False, repr=False)

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.W = rng.standard_normal((self.n_units, self.n_units)) / np.sqrt(self.n_units)

    def __call__(self, u: np.ndarray) -> np.ndarray:
        a = float(np.exp(-1.0 / self.tau))
        x = np.zeros(self.n_units)
        out = np.empty_like(u, dtype=float)
        drive = (1 - a) * (self.W @ u)
        for t in range(u.shape[1]):
            x = a * x + drive[:, t]
            out[:, t] = x
        return out


@dataclass
class ChaoticRateNetwork:
    """tau dx/dt = -x + g W tanh(x) + s u + sigma xi, with W fixed per seed and g a parameter.

    Boundary: W, g, tau, the input gain s and the internal noise sigma (part of the system).
    For N large and g > 1 the autonomous dynamics are chaotic (Sompolinsky et al. 1988) with
    exponentially decaying correlations; below g = 1 the fixed point is stable.
    ``observe`` selects a fixed subset of units for measurement.
    """

    n_units: int = 500
    gain: float = 1.5
    tau: float = 10.0
    input_scale: float = 0.3
    noise: float = 0.0
    seed: int = 0
    noise_seed: int = 1
    n_observe: int = 32
    unit_mask: np.ndarray | None = None
    W: np.ndarray = field(init=False, repr=False)
    observe: np.ndarray = field(init=False, repr=False)

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.W = rng.standard_normal((self.n_units, self.n_units)) / np.sqrt(self.n_units)
        self.observe = np.sort(rng.choice(self.n_units, self.n_observe, replace=False))
        self.x0 = rng.standard_normal(self.n_units) * 0.5

    def with_gain(self, gain: float) -> "ChaoticRateNetwork":
        c = ChaoticRateNetwork.__new__(ChaoticRateNetwork)
        c.__dict__.update(self.__dict__)
        c.gain = gain
        return c

    def __call__(self, u: np.ndarray, burn_in: int = 1000) -> np.ndarray:
        """``u`` has shape (n_units_in, T); only the first rows up to n_units are used, padded with zeros."""
        T = u.shape[1]
        drive = np.zeros((self.n_units, T + burn_in))
        k = min(u.shape[0], self.n_units)
        drive[:k, burn_in:] = self.input_scale * u[:k]
        if burn_in and k:
            drive[:k, :burn_in] = self.input_scale * u[:k, :min(burn_in, T)].mean(axis=1, keepdims=True)
        mask = np.ones(self.n_units) if self.unit_mask is None else self.unit_mask
        nr = np.random.default_rng(self.noise_seed)
        x = self.x0.copy()
        out = np.empty((self.n_observe, T))
        for t in range(T + burn_in):
            r = np.tanh(x) * mask
            x = x + (-x + self.gain * (self.W @ r) + drive[:, t] + self.noise * nr.standard_normal(self.n_units)) / self.tau
            if t >= burn_in:
                out[:, t - burn_in] = (np.tanh(x) * mask)[self.observe]
        return out


@dataclass
class OscillatorNetwork:
    """Coupled Stuart-Landau oscillators dz_k/dt = (mu + i w_k - |z_k|^2) z_k + K sum_j A_kj (z_j - z_k) + s u_k.

    Boundary: mu, the frequencies w_k, the coupling. For mu > 0 each unit has a stable limit
    cycle, so the autonomous activity is periodic: persistent, input-resistant and not scale-free.
    """

    n_units: int = 32
    mu: float = 1.0
    freq: float = 1.3
    freq_spread: float = 0.05
    coupling: float = 0.5
    fs: float = 250.0
    input_scale: float = 0.3
    seed: int = 0

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.w = 2 * np.pi * self.freq * (1 + self.freq_spread * rng.standard_normal(self.n_units))
        A = (rng.random((self.n_units, self.n_units)) < 0.2).astype(float)
        np.fill_diagonal(A, 0)
        self.A = A / (A.sum(axis=1, keepdims=True) + 1e-9)
        self.z0 = 0.1 * (rng.standard_normal(self.n_units) + 1j * rng.standard_normal(self.n_units))

    def __call__(self, u: np.ndarray, burn_in: int = 1000) -> np.ndarray:
        dt = 1.0 / self.fs
        T = u.shape[1]
        z = self.z0.copy()
        out = np.empty((self.n_units, T))
        for t in range(T + burn_in):
            drive = self.input_scale * u[: self.n_units, t - burn_in] if t >= burn_in else 0.0
            dz = (self.mu + 1j * self.w - np.abs(z) ** 2) * z + self.coupling * (self.A @ z - z) + drive
            z = z + dt * dz
            if t >= burn_in:
                out[:, t - burn_in] = z.real
        return out


@dataclass
class HeterogeneousTimescaleNetwork:
    """Leaky units tau_i dx_i/dt = -x_i + s u_i + xi_i with tau_i log-uniform over [tau_min, tau_max]
    and internal white noise xi_i. Boundary: the tau_i and the internal noise (part of the system).

    Each observed channel is the sum of a random subset of units, so its autonomous activity is a
    superposition of relaxation processes with log-uniform time constants, which has a 1/f spectrum
    and DFA exponent near 1 for scales inside [tau_min, tau_max] (the Bernamont / Granger mechanism).
    Scale-free for a known reason, within a known band.
    """

    n_units: int = 400
    tau_min: float = 2.0
    tau_max: float = 2000.0
    noise: float = 1.0
    input_scale: float = 0.3
    n_observe: int = 32
    units_per_channel: int = 50
    seed: int = 0
    noise_seed: int = 1

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.taus = np.exp(rng.uniform(np.log(self.tau_min), np.log(self.tau_max), self.n_units))
        self.readout = np.zeros((self.n_observe, self.n_units))
        for c in range(self.n_observe):
            idx = rng.choice(self.n_units, self.units_per_channel, replace=False)
            self.readout[c, idx] = 1.0 / np.sqrt(self.taus[idx])

    def __call__(self, u: np.ndarray, burn_in: int = 4000) -> np.ndarray:
        T = u.shape[1]
        a = np.exp(-1.0 / self.taus)
        gain = np.sqrt(1 - a**2)  # unit-variance stationary state per unit
        nr = np.random.default_rng(self.noise_seed)
        x = np.zeros(self.n_units)
        out = np.empty((self.n_observe, T))
        k = min(u.shape[0], self.n_units)
        for t in range(T + burn_in):
            drive = np.zeros(self.n_units)
            if t >= burn_in:
                drive[:k] = self.input_scale * u[:k, t - burn_in]
            x = a * x + gain * (self.noise * nr.standard_normal(self.n_units) + drive)
            if t >= burn_in:
                out[:, t - burn_in] = self.readout @ x
        return out


@dataclass
class CriticalBranchingNetwork:
    """Probabilistic branching network: each active unit activates each of k random targets with
    probability sigma / k at the next step; units also activate spontaneously with rate h.
    Boundary: the wiring, sigma and h. The population activity has correlation time
    -1 / ln(sigma), which diverges as sigma -> 1, giving near-scale-free fluctuations over a
    range bounded by that time and by the system size. Observed channels are activity counts
    in random subpopulations, smoothed over ``bin`` steps.
    """

    n_units: int = 2000
    sigma: float = 0.98
    k: int = 10
    h: float = 1e-3
    n_observe: int = 32
    bin: int = 1
    seed: int = 0
    noise_seed: int = 1

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.targets = np.array([rng.choice(self.n_units, self.k, replace=False) for _ in range(self.n_units)])
        self.groups = [rng.choice(self.n_units, self.n_units // self.n_observe, replace=False) for _ in range(self.n_observe)]

    def __call__(self, u: np.ndarray, burn_in: int = 2000) -> np.ndarray:
        """Input rows are treated as per-unit spontaneous-rate modulation: rate = h (1 + |u|)."""
        T = u.shape[1]
        nr = np.random.default_rng(self.noise_seed)
        p = self.sigma / self.k
        active = nr.random(self.n_units) < 0.01
        out = np.empty((self.n_observe, T))
        k = min(u.shape[0], self.n_units)
        for t in range(T + burn_in):
            rate = np.full(self.n_units, self.h)
            if t >= burn_in and k:
                rate[:k] = self.h * (1 + np.abs(u[:k, t - burn_in]))
            nxt = nr.random(self.n_units) < rate
            src = np.flatnonzero(active)
            if src.size:
                tg = self.targets[src].ravel()
                hit = nr.random(tg.size) < p
                nxt[tg[hit]] = True
            active = nxt
            if t >= burn_in:
                out[:, t - burn_in] = [active[g].sum() for g in self.groups]
        if self.bin > 1:
            nb = T // self.bin
            out = out[:, : nb * self.bin].reshape(self.n_observe, nb, self.bin).sum(axis=2)
        return out.astype(float)


# ---- controls/decomposition.py ----
"""Variance decomposition of a measurement into text/input-associated and
variant-associated parts (the cross-replay regression, generalised).

The quantity of interest is the increment in explained variance from adding
a factor to a base regression:

    dR2(term | base) = R2(base + term) - R2(base).

For a temporal or spectral summary of a replayed trajectory there are two
such increments worth reporting, and the cross-replay notebook reports only
the first:

* ``generation`` increment: add the generation-variant factor to a base that
  already contains the replay-variant factor. This asks whether the identity
  of the variant that produced the input explains anything beyond the
  measured input statistics.
* ``replay`` increment: add the replay-variant factor to a base that already
  contains the generation-variant factor. This asks whether the variant that
  processed a fixed input changes the measurement, which is the question
  "is this measurement sensitive to the model when the input is held fixed".

``both_increments`` returns the two together so that neither can be quoted
without the other.
"""


from dataclasses import dataclass, field

import numpy as np
import pandas as pd


def _ols_r2(df: pd.DataFrame, formula: str) -> float:
    import warnings

    import statsmodels.formula.api as smf

    with warnings.catch_warnings():
        # nested factors (an input generated by one variant) make the design rank-deficient by
        # construction; R2 is unaffected, so the warning is noise here
        warnings.simplefilter("ignore")
        return float(smf.ols(formula, data=df).fit().rsquared)


def _loo_cv_r2(df: pd.DataFrame, formula: str, group_col: str) -> float:
    """Leave-one-group-out cross-validated R2 of a formula (no CV on factors of the held-out group)."""
    import statsmodels.formula.api as smf

    import warnings

    y_col = formula.split("~")[0].strip()
    preds = np.full(len(df), np.nan)
    for g in df[group_col].unique():
        train = df[df[group_col] != g]
        test = df[df[group_col] == g]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = smf.ols(formula, data=train).fit()
        try:
            preds[df.index.get_indexer(test.index)] = model.predict(test)
        except Exception:
            continue
    y = df[y_col].to_numpy(dtype=float)
    ok = np.isfinite(preds)
    if ok.sum() < 3:
        return float("nan")
    ss_res = float(np.sum((y[ok] - preds[ok]) ** 2))
    ss_tot = float(np.sum((y[ok] - y[ok].mean()) ** 2)) + 1e-18
    return 1.0 - ss_res / ss_tot


@dataclass
class Increment:
    metric: str
    term: str
    base_formula: str
    full_formula: str
    r2_base: float
    r2_full: float
    n_rows: int

    @property
    def delta_r2(self) -> float:
        return self.r2_full - self.r2_base


def increment(df: pd.DataFrame, metric: str, base_terms: list[str], term: str) -> Increment:
    """dR2 from adding ``term`` to a regression of ``metric`` on ``base_terms``."""
    base = f"{metric} ~ " + (" + ".join(base_terms) if base_terms else "1")
    full = base + f" + {term}"
    d = df.dropna(subset=[metric])
    return Increment(metric, term, base, full, _ols_r2(d, base), _ols_r2(d, full), len(d))


@dataclass
class BothIncrements:
    metric: str
    generation: Increment
    replay: Increment
    text_only_r2: float = float("nan")
    text_only_cv_r2: float = float("nan")
    notes: list[str] = field(default_factory=list)

    def to_row(self) -> dict:
        return {
            "metric": self.metric,
            "dR2_generation": self.generation.delta_r2,
            "dR2_replay": self.replay.delta_r2,
            "R2_text_only": self.text_only_r2,
            "R2_text_only_cv": self.text_only_cv_r2,
            "R2_full": self.generation.r2_full,
            "n_rows": self.generation.n_rows,
        }


def both_increments(
    df: pd.DataFrame,
    metric: str,
    text_cols: list[str],
    generation_col: str = "generation_variant",
    replay_col: str = "replay_variant",
    design_cols: tuple[str, ...] = ("input_id",),
    cv_group: str | None = "input_id",
) -> BothIncrements:
    """Generation-variant and replay-variant increments for one measurement.

    ``df`` has one row per (input, replay variant) pair with the measurement in
    ``metric``, the measured input statistics in ``text_cols`` (may be empty),
    the identity of the variant that produced the input in ``generation_col``,
    the identity of the variant that processed it in ``replay_col`` and any
    design factors (prompt, temperature, subject) in ``design_cols``.
    """
    text_terms = list(text_cols)
    design_terms = [f"C({c})" for c in design_cols]
    gen_term, rep_term = f"C({generation_col})", f"C({replay_col})"
    gen = increment(df, metric, text_terms + design_terms + [rep_term], gen_term)
    rep = increment(df, metric, text_terms + design_terms + [gen_term], rep_term)
    out = BothIncrements(metric, gen, rep)
    d = df.dropna(subset=[metric])
    if text_terms:
        text_formula = f"{metric} ~ " + " + ".join(text_terms)
        out.text_only_r2 = _ols_r2(d, text_formula)
        if cv_group is not None and d[cv_group].nunique() >= 3:
            out.text_only_cv_r2 = _loo_cv_r2(d, text_formula, cv_group)
    if d[replay_col].nunique() < 2:
        out.notes.append("single replay variant: replay increment is zero by construction")
    if d[generation_col].nunique() < 2:
        out.notes.append("single generation variant: generation increment is zero by construction")
    return out


def pairwise_increment(
    df: pd.DataFrame,
    metric: str,
    text_cols: list[str],
    perturbation_col: str = "perturbation",
    generation_col: str = "generation_variant",
    design_cols: tuple[str, ...] = ("input_id",),
) -> Increment:
    """Perturbation-identity increment for a pairwise (intact-versus-perturbed) measurement.

    Reproduces the cross-replay notebook's step 3: base contains the text
    statistics, design factors and the generation variant; the perturbation
    label is added. Note that for a measurement *defined* as an
    intact-versus-perturbed distance the perturbation label is nearly the
    only remaining source of variance, so a large increment here is not
    comparable to the increments of ``both_increments``.
    """
    base = list(text_cols) + [f"C({c})" for c in design_cols] + [f"C({generation_col})"]
    return increment(df, metric, base, f"C({perturbation_col})")


# ---- controls/permutation.py ----
"""Design-respecting permutation null for an increment in explained variance.

Labels are permuted within blocks so that the repeated-measurement structure
of a replay design is preserved: a generation label is permuted at the level
of the generated input (all replays of one input receive the same permuted
label), and a replay label is permuted within each input (the inputs keep
their identity, the processing variants are shuffled among their replays).
"""


from dataclasses import dataclass

import numpy as np
import pandas as pd



@dataclass
class PermutationResult:
    observed: float
    null: np.ndarray
    p_value: float
    n_permutations: int

    @property
    def null_95th(self) -> float:
        return float(np.percentile(self.null, 95))


def _permute_within(df: pd.DataFrame, label_col: str, block_cols: list[str], rng: np.random.Generator,
                    unit_col: str | None = None) -> pd.DataFrame:
    """Permute ``label_col`` within blocks. With ``unit_col``, labels are
    permuted across units (all rows of a unit keep one label)."""
    out = df.copy()
    if unit_col is None:
        for _, idx in out.groupby(block_cols, sort=False).indices.items():
            idx = np.asarray(idx)
            out.iloc[idx, out.columns.get_loc(label_col)] = rng.permutation(out[label_col].to_numpy()[idx])
        return out
    cols = list(dict.fromkeys([unit_col, label_col, *block_cols]))
    if label_col in block_cols:
        raise ValueError("the permuted label cannot also be a blocking factor")
    units = out[cols].drop_duplicates(subset=[unit_col]).set_index(unit_col)
    for _, idx in units.groupby(block_cols, sort=False).indices.items():
        idx = np.asarray(idx)
        units.iloc[idx, units.columns.get_loc(label_col)] = rng.permutation(units[label_col].to_numpy()[idx])
    out[label_col] = out[unit_col].map(units[label_col])
    return out


def permutation_null(
    df: pd.DataFrame,
    metric: str,
    base_terms: list[str],
    label_col: str,
    block_cols: list[str],
    unit_col: str | None = None,
    n_permutations: int = 1000,
    seed: int = 0,
) -> PermutationResult:
    """Permutation null of dR2 from adding ``C(label_col)`` to ``base_terms``.

    The empirical p-value uses the plus-one correction, so its minimum is
    1 / (n_permutations + 1).
    """
    rng = np.random.default_rng(seed)
    term = f"C({label_col})"
    observed = increment(df, metric, base_terms, term).delta_r2
    null = np.empty(n_permutations)
    for k in range(n_permutations):
        shuffled = _permute_within(df, label_col, block_cols, rng, unit_col)
        null[k] = increment(shuffled, metric, base_terms, term).delta_r2
    p = (1 + int(np.sum(null >= observed))) / (1 + n_permutations)
    return PermutationResult(observed, null, float(p), n_permutations)


# ---- controls/bootstrap.py ----
"""Cluster bootstrap intervals for increments and for subject-level AUC."""


import numpy as np
import pandas as pd



def cluster_bootstrap_increment(
    df: pd.DataFrame,
    metric: str,
    base_terms: list[str],
    term: str,
    cluster_col: str,
    n_boot: int = 500,
    seed: int = 0,
    ci: tuple[float, float] = (2.5, 97.5),
) -> tuple[float, float]:
    """Percentile CI of dR2 by resampling clusters (prompts, subjects) with replacement."""
    rng = np.random.default_rng(seed)
    clusters = df[cluster_col].unique()
    groups = {c: df[df[cluster_col] == c] for c in clusters}
    boots = []
    for _ in range(n_boot):
        draw = rng.choice(clusters, size=len(clusters), replace=True)
        parts = []
        for k, c in enumerate(draw):
            g = groups[c].copy()
            g[cluster_col] = f"{c}__b{k}"
            parts.append(g)
        sample = pd.concat(parts, ignore_index=True)
        try:
            boots.append(increment(sample, metric, base_terms, term).delta_r2)
        except Exception:
            continue
    if not boots:
        return float("nan"), float("nan")
    lo, hi = np.percentile(boots, ci)
    return float(lo), float(hi)


def auc(positive: np.ndarray, negative: np.ndarray) -> float:
    """Mann-Whitney AUC: P(score_positive > score_negative) with ties counted half."""
    positive = np.asarray(positive, dtype=float)
    negative = np.asarray(negative, dtype=float)
    diff = positive[:, None] - negative[None, :]
    return float((np.sum(diff > 0) + 0.5 * np.sum(diff == 0)) / diff.size)


def bootstrap_auc_paired(
    v_positive: np.ndarray, v_negative: np.ndarray, n_boot: int = 2000, seed: int = 42
) -> tuple[float, float, float]:
    """Subject-level AUC with a percentile CI from resampling paired subjects (paper Section 6.9)."""
    v1 = np.asarray(v_positive, dtype=float)
    v2 = np.asarray(v_negative, dtype=float)
    rng = np.random.default_rng(seed)
    n = v1.size
    point = auc(v1, v2)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        boots.append(auc(v1[idx], v2[idx]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, float(lo), float(hi)


# ---- controls/paired.py ----
"""Paired, subject-level statistics (paper Section 6.9): Friedman omnibus,
Bonferroni-corrected Wilcoxon post-hoc tests, matched-pairs rank-biserial
effect size and subject-level AUC benchmarking."""


from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, rankdata, wilcoxon



def rank_biserial(v1: np.ndarray, v2: np.ndarray) -> float:
    """Matched-pairs rank-biserial correlation of v1 - v2 (+1 when every pair has v1 > v2)."""
    d = np.asarray(v1, dtype=float) - np.asarray(v2, dtype=float)
    d = d[d != 0]
    if d.size == 0:
        return 0.0
    r = rankdata(np.abs(d))
    wp, wn = r[d > 0].sum(), r[d < 0].sum()
    return float((wp - wn) / (wp + wn)) if (wp + wn) > 0 else 0.0


@dataclass
class PairedResult:
    friedman_chi2: float
    friedman_p: float
    n_complete: int
    pairs: dict[tuple[str, str], dict] = field(default_factory=dict)


def subject_level_table(df: pd.DataFrame, value_col: str, subject_col: str = "subject",
                        stage_col: str = "stage") -> pd.DataFrame:
    """Subject-by-stage table of per-subject means (the paper's unit of analysis)."""
    return df.groupby([subject_col, stage_col])[value_col].mean().unstack(stage_col)


def paired_stage_tests(piv: pd.DataFrame, stages: list[str], pairs: list[tuple[str, str]],
                       alpha: float = 0.05) -> PairedResult:
    """Friedman omnibus over ``stages`` and Bonferroni-corrected Wilcoxon tests over ``pairs``."""
    present = [s for s in stages if s in piv.columns]
    complete = piv[present].dropna()
    chi2 = p = float("nan")
    if len(complete) >= 3 and len(present) >= 3:
        chi2, p = friedmanchisquare(*[complete[s].to_numpy() for s in present])
    out = PairedResult(float(chi2), float(p), len(complete))
    bonf = alpha / max(len(pairs), 1)
    for s1, s2 in pairs:
        if s1 not in piv.columns or s2 not in piv.columns:
            continue
        sh = piv[[s1, s2]].dropna()
        v1, v2 = sh[s1].to_numpy(), sh[s2].to_numpy()
        if len(sh) < 3 or np.allclose(v1, v2):
            continue
        stat, pw = wilcoxon(v1, v2)
        out.pairs[(s1, s2)] = {
            "n": len(sh),
            "wilcoxon_p": float(pw),
            "bonferroni_threshold": bonf,
            "significant": bool(pw < bonf),
            "rank_biserial": rank_biserial(v1, v2),
            "mean_1": float(v1.mean()),
            "mean_2": float(v2.mean()),
        }
    return out


def subject_auc_benchmark(df: pd.DataFrame, pairs: list[tuple[str, str]], metrics: list[str],
                          subject_col: str = "subject", stage_col: str = "stage",
                          n_boot: int = 2000, seed: int = 42) -> pd.DataFrame:
    """Subject-level AUC with bootstrap CI for every (stage pair, metric) (paper Table 3)."""
    rows = []
    for metric in metrics:
        piv = subject_level_table(df, metric, subject_col, stage_col)
        for s1, s2 in pairs:
            if s1 not in piv.columns or s2 not in piv.columns:
                continue
            sh = piv[[s1, s2]].dropna()
            if len(sh) < 4:
                continue
            a, lo, hi = bootstrap_auc_paired(sh[s1].to_numpy(), sh[s2].to_numpy(), n_boot, seed)
            rows.append({"pair": f"{s1} vs {s2}", "metric": metric, "auc": a, "ci_lo": lo, "ci_hi": hi, "n": len(sh)})
    return pd.DataFrame(rows)


# ---- controls/replay.py ----
"""Two-direction input control (the cross-replay design, made generic).

A claim that a dynamical quantity is a property of a system rather than of
its input has to survive two manipulations:

* hold the input fixed and vary the system (replay direction), and
* hold the system fixed and vary the input (input direction).

``ReplayHarness`` takes a dictionary of systems (any callable that maps an
input to an activation array of shape (channels, time)) and a dictionary of
inputs, runs every system on every input, computes the chosen measurements
on each activation array, and returns a long table with one row per
(input, system). ``decompose`` then reports, for every measurement, the
increment in explained variance attributable to the system identity with
the input held fixed and to the input identity with the system held fixed.
Nothing here depends on a particular framework: a PyTorch model, a NumPy
recurrent network or an EEG "system" (a subject) all fit the interface.
"""


from dataclasses import dataclass, field
from typing import Callable, Mapping

import numpy as np
import pandas as pd


ActivationFn = Callable[[np.ndarray], np.ndarray]
MeasureFn = Callable[[np.ndarray], Mapping[str, float]]


def psi_component_measures(config: AuditConfig) -> MeasureFn:
    """Measurement function returning the raw Psi components of an activation array."""

    def _measure(X: np.ndarray) -> dict[str, float]:
        c = compute_components(X, config)
        return {"alpha_dfa": c.alpha_dfa, "i_phi_a": c.i_phi_a, "lz": c.lz, "d": c.d, "m": c.m}

    return _measure


@dataclass
class DirectionResult:
    metric: str
    dR2_system_given_input: float
    dR2_input_given_system: float
    r2_full: float
    n_rows: int
    dominant: str

    def to_row(self) -> dict:
        return {
            "metric": self.metric,
            "dR2_system|input": self.dR2_system_given_input,
            "dR2_input|system": self.dR2_input_given_system,
            "R2_full": self.r2_full,
            "n_rows": self.n_rows,
            "dominant": self.dominant,
        }


@dataclass
class ReplayHarness:
    systems: Mapping[str, ActivationFn]
    inputs: Mapping[str, np.ndarray]
    measure: MeasureFn
    input_features: Mapping[str, Mapping[str, float]] | None = None
    table: pd.DataFrame = field(default_factory=pd.DataFrame, repr=False)

    def run(self, verbose: bool = False) -> pd.DataFrame:
        rows = []
        for in_id, x in self.inputs.items():
            feats = dict(self.input_features.get(in_id, {})) if self.input_features else {}
            for sys_id, fn in self.systems.items():
                act = np.asarray(fn(x), dtype=float)
                row = {"input_id": in_id, "system_id": sys_id, **feats, **self.measure(act)}
                rows.append(row)
                if verbose:
                    print(f"  {sys_id:>16s} x {in_id:<16s} done")
        self.table = pd.DataFrame(rows)
        return self.table

    def decompose(self, metrics: list[str] | None = None, input_feature_cols: list[str] | None = None) -> pd.DataFrame:
        """Both-direction increments for each measurement.

        ``input_feature_cols`` are measured input statistics entered as
        covariates in both directions, analogous to the generated-text
        statistics of the cross-replay paper. When none are given the input
        identity factor absorbs all input variation.
        """
        if self.table.empty:
            self.run()
        df = self.table
        reserved = {"input_id", "system_id", *(input_feature_cols or [])}
        metrics = metrics or [c for c in df.columns if c not in reserved and np.issubdtype(df[c].dtype, np.number)]
        feats = list(input_feature_cols or [])
        results = []
        for m in metrics:
            d = df.dropna(subset=[m])
            if d[m].nunique() < 2:
                continue
            sys_inc = increment(d, m, feats + ["C(input_id)"], "C(system_id)")
            in_inc = increment(d, m, feats + ["C(system_id)"], "C(input_id)")
            a, b = sys_inc.delta_r2, in_inc.delta_r2
            dominant = "system" if a > b else "input"
            results.append(DirectionResult(m, a, b, sys_inc.r2_full, len(d), dominant).to_row())
        return pd.DataFrame(results)


def endogeneity_ratio(decomp: pd.DataFrame) -> pd.Series:
    """dR2_system|input / (dR2_system|input + dR2_input|system) per metric; near 1 means system-driven."""
    a = decomp["dR2_system|input"].to_numpy(dtype=float)
    b = decomp["dR2_input|system"].to_numpy(dtype=float)
    return pd.Series(a / (a + b + 1e-12), index=decomp["metric"])


# ---- controls/attribution.py ----
"""Attribution of a measurement to system, input and their interaction, and
conditional sensitivity to a matched perturbation.

The additive two-direction increments of ``decomposition`` answer "does the
system matter with the input fixed" and the converse. A general account
also needs the interaction, because a system that transforms its input in
a system-specific way shows up as a large system-by-input term, which is a
finding rather than a failure. With replicate runs per (system, input)
cell the model

    z = mu + A_system + B_input + C_system:input + noise

is identifiable and the three components can be reported as shares of
explained variance. Without replicates the interaction is confounded with
the residual and is reported as not identifiable.

``conditional_sensitivity`` answers a different question from attribution:
under a matched perturbation of one system, how consistently does the
measurement move across inputs? A measurement can be input-dominated in
attribution terms and still move the same way under damage on every input.
"""


from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


def _r2(df: pd.DataFrame, formula: str) -> float:
    return _ols_r2(df, formula)


@dataclass
class Attribution:
    metric: str
    share_system: float
    share_input: float
    share_interaction: float
    share_residual: float
    dR2_system_given_input: float
    dR2_input_given_system: float
    interaction_identifiable: bool
    n_rows: int
    n_systems: int
    n_inputs: int
    replicates_per_cell: float

    def to_row(self) -> dict:
        return asdict(self)


def attribute(df: pd.DataFrame, metric: str, system_col: str = "system_id", input_col: str = "input_id",
              covariates: list[str] | None = None) -> Attribution:
    """Shares of variance for system, input and interaction (sequential, system first then input,
    which is order-independent for a balanced design) plus the two-direction increments."""
    d = df.dropna(subset=[metric])
    cov = list(covariates or [])
    base_terms = cov
    f_base = f"{metric} ~ " + (" + ".join(base_terms) if base_terms else "1")
    f_sys = f_base + f" + C({system_col})"
    f_add = f_sys + f" + C({input_col})"
    f_inp = f_base + f" + C({input_col})"
    f_int = f_add + f" + C({system_col}):C({input_col})"
    r_base, r_sys, r_add, r_inp = _r2(d, f_base), _r2(d, f_sys), _r2(d, f_add), _r2(d, f_inp)
    cells = d.groupby([system_col, input_col]).size()
    reps = float(cells.mean())
    identifiable = bool((cells > 1).all())
    r_int = _r2(d, f_int) if identifiable else float("nan")
    share_sys = r_sys - r_base
    share_inp = r_add - r_sys
    share_int = (r_int - r_add) if identifiable else float("nan")
    resid = 1.0 - (r_int if identifiable else r_add)
    return Attribution(
        metric=metric, share_system=float(share_sys), share_input=float(share_inp), share_interaction=float(share_int),
        share_residual=float(resid), dR2_system_given_input=float(r_add - r_inp), dR2_input_given_system=float(r_add - r_sys),
        interaction_identifiable=identifiable, n_rows=len(d), n_systems=int(d[system_col].nunique()),
        n_inputs=int(d[input_col].nunique()), replicates_per_cell=reps,
    )


@dataclass
class ConditionalSensitivity:
    metric: str
    reference: str
    target: str
    n_pairs: int
    mean_delta: float
    sd_delta: float
    effect_size_dz: float
    sign_consistency: float
    delta_over_between_input_sd: float
    wilcoxon_p: float

    def to_row(self) -> dict:
        return asdict(self)


def conditional_sensitivity(df: pd.DataFrame, metric: str, unit_col: str, variant_col: str,
                            reference: str, target: str) -> ConditionalSensitivity:
    """Paired change in ``metric`` from ``reference`` to ``target`` across units (inputs), with the
    paired effect size d_z, the fraction of units moving in the majority direction, the mean change
    expressed in units of the between-input SD of the reference, and a Wilcoxon signed-rank p."""
    piv = df.pivot_table(index=unit_col, columns=variant_col, values=metric, aggfunc="mean")
    if reference not in piv or target not in piv:
        raise KeyError(f"variants {reference!r} and {target!r} not both present in {variant_col!r}")
    sh = piv[[reference, target]].dropna()
    delta = (sh[target] - sh[reference]).to_numpy(dtype=float)
    n = delta.size
    sd = float(delta.std(ddof=1)) if n > 1 else float("nan")
    dz = float(delta.mean() / (sd + 1e-18)) if n > 1 else float("nan")
    sign = float(max(np.mean(delta > 0), np.mean(delta < 0))) if n else float("nan")
    between = float(sh[reference].std(ddof=1)) if n > 1 else float("nan")
    try:
        p = float(wilcoxon(delta).pvalue) if n >= 5 and not np.allclose(delta, 0) else float("nan")
    except ValueError:
        p = float("nan")
    return ConditionalSensitivity(metric, reference, target, n, float(delta.mean()), sd, dz, sign,
                                  float(delta.mean() / (between + 1e-18)), p)


print('cinn-audit library loaded (inline).')

In [ ]:
import os, time, json, shutil, warnings
warnings.filterwarnings("ignore")
for sub in ("E2v2", "E4v2"):
    os.makedirs(os.path.join(RESULTS_DIR, sub), exist_ok=True)

def out_path(name, ext):
    return os.path.join(RESULTS_DIR, f"{name}.{ext}")

def save_df(df, name):
    df.to_csv(out_path(name, "csv"), index=False)

def save_json(obj, name):
    with open(out_path(name, "json"), "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else (o.tolist() if hasattr(o, "tolist") else str(o)))

print("common helpers ready")

## E2v2. Attribution, sensitivity and validity on reference systems

Six system families whose generating mechanisms are known in advance, so that the correct answer to each of the
three questions is known before the analysis is run.

| Family | Distinction tested | Settings |
|---|---|---|
| Feedforward filter | Structure inherited from the input, no state of its own | random mixing, tanh, band-pass in one of three bands |
| Leaky network | Finite memory without persistence | $x_{t+1} = a x_t + (1-a) W u_t$, $\tau$ = 5, 20 or 80 steps |
| Chaotic rate network | Persistence without scaling, and regime versus matrix | $N$ = 500, three fixed matrices each swept across six gains |
| Stuart–Landau oscillators | Periodic persistence, the deliberate negative for scaling | 32 units near 1.3 Hz, coupling 0.5 |
| Heterogeneous timescales | Scaling by superposition of relaxation processes | 400 leaky units, $\tau$ log-uniform over three decades, internal noise |
| Critical branching | Scaling near a critical point | 2000 binary units, $\sigma$ = 0.9, 0.98 or 0.995 |

Every system is driven by the same eight band-limited noise inputs with two replicates that differ only in the
internal-noise seed, which makes the system, input and interaction shares identifiable, and every system is run
three times on zero input for the persistence test. The validity checks are applied to each channel before any
scaling claim is made. Conditional sensitivity is measured on the $g = 2$ chaotic network under unit removal and
weight noise with the inputs held fixed.

In [ ]:
# ---- E2v2: attribution on reference systems with known regimes ---------------------------------
t0 = time.time()
N_IN = 32

def make_inputs(n, seed):
    rng = np.random.default_rng(seed)
    bands = [(1, 8), (4, 20), (8, 40), (1, 40), (13, 60), (2, 12), (6, 30), (1, 60)]
    inputs = {}
    for k in range(n):
        lo, hi = bands[k % len(bands)]
        x = bandpass(rng.standard_normal((N_IN, E2_STEPS + 500)), E2_FS, lo, hi)[:, 500:]
        x /= x.std(axis=1, keepdims=True)
        inputs[f"in{k:02d}_{lo}-{hi}Hz"] = x * (0.5 + 1.0 * rng.random())
    return inputs

inputs2 = make_inputs(E2_N_INPUTS, SEED)
zero_in = np.zeros((N_IN, E2_ZERO_STEPS))

# -- the systems. Each entry: (family, system id, constructor taking noise_seed) --
def systems_table():
    rows = []
    for s, band in enumerate([(2, 30), (4, 45), (8, 60)]):
        rows.append(("feedforward", f"ff_{band[0]}-{band[1]}Hz", lambda ns, b=band, s=s: FeedforwardFilter(N_IN, E2_FS, b, seed=s)))
    for tau in E2_LEAKY_TAUS:
        rows.append(("leaky", f"leaky_tau{tau:g}", lambda ns, tau=tau: LeakyNetwork(N_IN, tau=tau, seed=0)))
    for seed in E2_CHAOTIC_SEEDS:
        for g in E2_CHAOTIC_GAINS:
            rows.append(("chaotic", f"chaotic_W{seed}_g{g}", lambda ns, g=g, seed=seed: ChaoticRateNetwork(E2_CHAOTIC_N, gain=g, noise=0.02, seed=100 + seed, noise_seed=ns)))
    for seed in range(3 if not DRY_RUN else 1):
        rows.append(("oscillator", f"osc_{seed}", lambda ns, seed=seed: OscillatorNetwork(N_IN, fs=E2_FS, seed=seed)))
    for seed in range(3 if not DRY_RUN else 1):
        rows.append(("heterogeneous", f"hetero_{seed}", lambda ns, seed=seed: HeterogeneousTimescaleNetwork(seed=seed, noise_seed=ns)))
    for sg in E2_BRANCHING_SIGMAS:
        rows.append(("branching", f"branching_s{sg}", lambda ns, sg=sg: CriticalBranchingNetwork(sigma=sg, seed=0, noise_seed=ns)))
    return rows

SYSTEMS = systems_table()
print(f"{len(SYSTEMS)} systems x {len(inputs2)} inputs x {E2_REPLICATES} replicates")

cfg2 = BIOLCYB2026_SYNTHETIC.with_fs(E2_FS)

def measure(act):
    act = np.atleast_2d(act)
    act = act[:, :]
    z = zscore_channels(act)
    v = channel_scaling_validity(z, E2_FS, max_channels=8, n_surrogates=E2_N_SURROGATES)
    m, _ = band_metastability(z, E2_FS, cfg2.metastability)
    return {"variance": float(act.var(axis=1).mean()), "alpha_dfa": v["alpha"], "dfa_r2": v["r_squared"],
            "slope_short": v["slope_short"], "slope_long": v["slope_long"], "crossover": v["crossover"],
            "spectral_concentration": v["spectral_concentration"], "fraction_valid": v["fraction_valid"],
            "validity_flags": v["flags"], "spectral_slope": spectral_slope(z, E2_FS), "m": m}

# -- driven runs --
rows = []
for family, sid, make in SYSTEMS:
    for in_id, u in inputs2.items():
        for rep in range(E2_REPLICATES):
            sysfn = make(1000 + rep)
            act = sysfn(u)
            rows.append({"family": family, "system_id": sid, "input_id": in_id, "replicate": rep, **measure(act)})
    print(f"  driven  {sid:22s} done ({time.time()-t0:5.0f} s)")
driven = pd.DataFrame(rows); save_df(driven, "E2v2/driven_measurements")

# -- autonomous runs. Persistence is judged against the driven variance of the same system, so an
#    internal noise floor is not mistaken for autonomous activity. --
driven_var = driven.groupby("system_id")["variance"].mean()
rows = []
for family, sid, make in SYSTEMS:
    for rep in range(E2_ZERO_REPEATS):
        act = make(2000 + rep)(zero_in)
        p = persistence(act, variance_min=max(1e-6, 0.05 * float(driven_var.get(sid, 0.0))))
        row = {"family": family, "system_id": sid, "repeat": rep, **p.to_row()}
        if p.persistent:
            row.update(measure(act))
        rows.append(row)
    print(f"  zero-in {sid:22s} done ({time.time()-t0:5.0f} s)")
auto = pd.DataFrame(rows); save_df(auto, "E2v2/autonomous_measurements")

# -- attribution per family (and across the chaotic gain sweep within each matrix) --
METRICS = ["alpha_dfa", "spectral_slope", "m", "slope_short", "slope_long"]
att_rows = []
for family, sub in driven.groupby("family"):
    for met in METRICS:
        if sub[met].nunique() < 2 or sub["system_id"].nunique() < 2:
            continue
        a = attribute(sub, met); att_rows.append({"scope": family, **a.to_row()})
for seed in E2_CHAOTIC_SEEDS:
    sub = driven[driven["system_id"].str.startswith(f"chaotic_W{seed}_")]
    for met in METRICS:
        if sub[met].nunique() < 2:
            continue
        a = attribute(sub, met); att_rows.append({"scope": f"chaotic_gain_sweep_W{seed}", **a.to_row()})
attribution = pd.DataFrame(att_rows); save_df(attribution, "E2v2/attribution")

# -- inferential profile per system --
prof = []
for family, sid, _ in SYSTEMS:
    d = driven[driven["system_id"] == sid]; z = auto[auto["system_id"] == sid]
    fam_att = attribution[(attribution["scope"] == family) & (attribution["metric"] == "alpha_dfa")]
    prof.append({
        "family": family, "system_id": sid,
        "driven_variance": d["variance"].mean(), "driven_alpha": d["alpha_dfa"].mean(),
        "driven_alpha_sd_over_inputs": d.groupby("input_id")["alpha_dfa"].mean().std(),
        "driven_crossover": d["crossover"].mean(), "driven_fraction_valid": d["fraction_valid"].mean(),
        "autonomous_persistent": bool(z["persistent"].all()), "autonomous_variance": z["variance"].mean(),
        "autonomous_over_driven_variance": z["variance"].mean() / (d["variance"].mean() + 1e-12),
        "autonomous_alpha": z["alpha_dfa"].mean() if "alpha_dfa" in z else np.nan,
        "autonomous_concentration": z["spectral_concentration"].mean() if "spectral_concentration" in z else np.nan,
        "autonomous_fraction_valid": z["fraction_valid"].mean() if "fraction_valid" in z else np.nan,
        "autonomous_flags": ";".join(sorted(set(z["validity_flags"].dropna()))) if "validity_flags" in z else "",
        "family_share_system": fam_att["share_system"].iloc[0] if len(fam_att) else np.nan,
        "family_share_input": fam_att["share_input"].iloc[0] if len(fam_att) else np.nan,
        "family_share_interaction": fam_att["share_interaction"].iloc[0] if len(fam_att) else np.nan,
    })
profile = pd.DataFrame(prof); save_df(profile, "E2v2/inferential_profile")

# -- conditional sensitivity: chaotic g=2.0 intact vs unit-pruned vs weight-noised, fixed input --
sens_rows, cs_rows = [], []
base = ChaoticRateNetwork(E2_CHAOTIC_N, gain=2.0, noise=0.02, seed=100, noise_seed=1000)
rng_p = np.random.default_rng(SEED + 7)
mask = np.ones(E2_CHAOTIC_N); mask[rng_p.choice(E2_CHAOTIC_N, E2_CHAOTIC_N // 4, replace=False)] = 0.0
pruned = ChaoticRateNetwork.__new__(ChaoticRateNetwork); pruned.__dict__.update(base.__dict__); pruned.unit_mask = mask
noised = ChaoticRateNetwork.__new__(ChaoticRateNetwork); noised.__dict__.update(base.__dict__)
noised.W = base.W + 0.5 * base.W.std() * rng_p.standard_normal(base.W.shape)
for vname, sysfn in (("intact", base), ("unit_pruned", pruned), ("weight_noise", noised)):
    for in_id, u in inputs2.items():
        sens_rows.append({"variant": vname, "input_id": in_id, **measure(sysfn(u))})
sens = pd.DataFrame(sens_rows); save_df(sens, "E2v2/perturbation_measurements")
for met in ("alpha_dfa", "spectral_slope", "m"):
    for target in ("unit_pruned", "weight_noise"):
        cs_rows.append(conditional_sensitivity(sens, met, "input_id", "variant", "intact", target).to_row())
sens_tbl = pd.DataFrame(cs_rows); save_df(sens_tbl, "E2v2/conditional_sensitivity")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("\n=== E2v2 inferential profile")
print(profile.round(3).to_string(index=False))
print("\n=== E2v2 attribution (shares of variance; interaction identifiable only with internal noise)")
print(attribution[attribution["metric"].isin(["alpha_dfa", "m"])].round(3).to_string(index=False))
print("\n=== E2v2 conditional sensitivity of the g = 2.0 chaotic network under fixed input")
print(sens_tbl.round(3).to_string(index=False))
print(f"\nE2v2 done in {time.time()-t0:.0f} s")

## E4v2. Attribution and sensitivity across a Pythia scaling series

Pythia-410M, 1.4B and 2.8B share their architecture and training data, so the comparison across size is made under
shared conditions. Each model is compared with two altered versions of itself. Head pruning keeps a random quarter
of the attention heads in the last $\lceil L/6 \rceil$ layers and zeroes the output projection of the rest. Weight
noise adds Gaussian noise to every matrix of the same layers, with the fraction calibrated so that the increase in
negative log-likelihood is as close as possible to that caused by pruning. Perturbations are applied in place and
restored afterwards, so one copy of each model suffices on a T4.

Each variant generates text from ten prompts, and every text is replayed through every variant. Replay is what
separates a change in how a text is processed from a change in which text is produced. For the temporal
statistics (DFA exponent, autocorrelation time, spectral slope, effective rank, trajectory velocity) the notebook
reports the attribution shares for text and variant, the increments in both directions with a permutation null,
and the conditional sensitivity (paired change across texts, sign consistency, effect size). For the geometric
measures (CKA distance, Procrustes distance, attention divergence) it reports the increment for the identity of
the intervention. All three models run in 32-bit precision because the largest produces non-finite logits in
half precision.

In [ ]:
# ---- E4v2: cross-replay on a Pythia scaling series -------------------------------------------
import math, contextlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTNeoXConfig, GPTNeoXForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"
t0 = time.time()

def dtype_for(name):
    # float16 only for the models that would not fit a 16 GB card in float32; fp16 overflow in a perturbed
    # late layer is what produces non-finite logits, so the smaller models are kept in float32.
    if device != "cuda":
        return torch.float32
    size = {"pythia-410m": 4.1e8, "pythia-1.4b": 1.4e9, "pythia-2.8b": 2.8e9}.get(name.split("/")[-1], 1e9)
    return torch.float16 if size > E4_FP16_ABOVE_PARAMS else torch.float32

PROMPTS = [
    "The history of the printing press begins with",
    "In a small coastal town, the lighthouse keeper",
    "To configure the network interface, first",
    "The committee reviewed the proposal and concluded that",
    "A: Have you seen the new exhibition? B:",
    "Photosynthesis converts light energy into",
    "The recipe calls for two cups of flour and",
    "When the train finally arrived at the station,",
    "The main difference between the two algorithms is",
    "Every morning the market opened with",
][:E4_N_PROMPTS]

# -- architecture-generic accessors (GPT-2 and GPT-NeoX families) --
def blocks_of(m):
    return m.transformer.h if hasattr(m, "transformer") else m.gpt_neox.layers
def out_proj_of(block):
    return block.attn.c_proj if hasattr(block, "attn") else block.attention.dense
def is_conv1d(mod):
    return mod.__class__.__name__ == "Conv1D"
def late_layers(m):
    L = m.config.num_hidden_layers; k = math.ceil(L / 6)
    return list(range(L - k, L))

@contextlib.contextmanager
def perturbed(m, kind, seed, noise_fraction=None):
    # Modify the late layers in place and restore afterwards, so one copy of the model is enough.
    # The saved copies live on the CPU so a float32 2.8B model still fits a 16 GB card.
    noise_fraction = E4_NOISE_FRACTION if noise_fraction is None else noise_fraction
    LATE = late_layers(m); H = m.config.num_attention_heads; d = m.config.hidden_size // H
    saved = [(p, p.detach().to("cpu", copy=True)) for li in LATE for p in blocks_of(m)[li].parameters()]
    g = torch.Generator(device="cpu").manual_seed(seed)
    rng = np.random.default_rng(seed)
    try:
        with torch.no_grad():
            if kind == "head_pruned":
                keep = set(np.sort(rng.choice(H, max(1, math.ceil(E4_PRUNE_KEEP_FRACTION * H)), replace=False)).tolist())
                for li in LATE:
                    W = out_proj_of(blocks_of(m)[li]).weight
                    for h in range(H):
                        if h not in keep:
                            if is_conv1d(out_proj_of(blocks_of(m)[li])):
                                W[h * d:(h + 1) * d, :] = 0.0       # Conv1D: (in, out), head slice on rows
                            else:
                                W[:, h * d:(h + 1) * d] = 0.0       # Linear: (out, in), head slice on columns
            elif kind == "weight_noise":
                for li in LATE:
                    for p in blocks_of(m)[li].parameters():
                        if p.dim() == 2:
                            noise = torch.randn(p.shape, generator=g).to(p.device, p.dtype)
                            p.add_(noise_fraction * p.float().std().to(p.dtype) * noise)
            elif kind != "intact":
                raise ValueError(kind)
        yield m
    finally:
        try:
            with torch.no_grad():
                for p, s in saved:
                    p.copy_(s.to(p.device))
        except Exception as restore_exc:  # a dead CUDA context must not mask the original error
            print(f"[perturbed] restore failed: {type(restore_exc).__name__}: {restore_exc}")

def load_lm(name):
    if name == "dry-run-neox":
        cfg = GPTNeoXConfig(num_hidden_layers=6, num_attention_heads=4, hidden_size=64, intermediate_size=128,
                            vocab_size=512, max_position_embeddings=256, attn_implementation="eager")
        torch.manual_seed(0); return GPTNeoXForCausalLM(cfg).to(device).eval(), None
    tok = AutoTokenizer.from_pretrained(name, cache_dir=HF_CACHE)
    dtype = dtype_for(name)
    # device_map loads the weights straight onto the GPU instead of staging a full float32 copy in the
    # runtime's RAM, which is what breaks a 12 GB Colab machine on the 2.8B model.
    load_kw = dict(attn_implementation="eager", cache_dir=HF_CACHE, torch_dtype=dtype, low_cpu_mem_usage=True,
                   device_map=(device if device == "cuda" else None))
    try:
        m = AutoModelForCausalLM.from_pretrained(name, **load_kw)
    except Exception as exc:
        print(f"cached weights for {name} unreadable ({type(exc).__name__}); clearing cache entry and re-downloading")
        import glob
        for dpath in glob.glob(os.path.join(HF_CACHE, "**", "models--" + name.replace("/", "--")), recursive=True):
            shutil.rmtree(dpath, ignore_errors=True)
        m = AutoModelForCausalLM.from_pretrained(name, force_download=True, **load_kw)
    m = m.to(device).eval()
    if not logits_finite(m, tok) and dtype != torch.float32:
        print(f"   {name}: non-finite logits in {dtype}; reloading in float32")
        del m
        if device == "cuda":
            torch.cuda.empty_cache()
        load_kw["torch_dtype"] = torch.float32
        m = AutoModelForCausalLM.from_pretrained(name, **load_kw).to(device).eval()
    assert logits_finite(m, tok), f"{name}: non-finite logits even in float32; stopping rather than measuring garbage"
    print(f"   loaded {name} in {next(m.parameters()).dtype}")
    return m, tok

def logits_finite(m, tok):
    ids = encode(tok, PROMPTS[0])
    with torch.no_grad():
        out = m(ids).logits
    return bool(torch.isfinite(out).all())

def encode(tok, prompt):
    if tok is None:
        g = torch.Generator().manual_seed(abs(hash(prompt)) % 2**31)
        return torch.randint(0, 512, (1, 12), generator=g).to(device)
    return tok(prompt, return_tensors="pt").input_ids.to(device)

def generate(m, tok, ids, seed):
    # remove_invalid_values replaces non-finite logits before sampling, which otherwise raises a device-side
    # assert in torch.multinomial and kills the CUDA context. Whether a variant produced non-finite logits is
    # recorded separately by ``replay`` (nonfinite_fraction), so this does not hide damage, it only keeps the run alive.
    torch.manual_seed(seed)
    with torch.no_grad():
        return m.generate(ids, do_sample=True, temperature=E4_TEMPERATURE, top_k=0, top_p=1.0, max_new_tokens=E4_NEW_TOKENS,
                          pad_token_id=(tok.eos_token_id if tok is not None else 0), remove_invalid_values=True)

def replay(m, ids, LATE):
    with torch.no_grad():
        o = m(ids, output_hidden_states=True, output_attentions=True)
        hs = [o.hidden_states[li + 1][0].float().cpu().numpy() for li in LATE]
        at = [o.attentions[li][0].float().cpu().numpy() for li in LATE]
        logits = o.logits[0, :-1].float()
        nonfinite = float((~torch.isfinite(logits)).any(dim=-1).float().mean())
        logp = torch.log_softmax(torch.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4), -1).gather(1, ids[0, 1:, None]).squeeze(1)
    hs = [np.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0) for h in hs]
    at = [np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0) for a in at]
    return hs, at, float(-logp.mean()), nonfinite

def text_stats(ids):
    t = ids[0].tolist(); n = len(t)
    def distinct(k): grams = [tuple(t[i:i+k]) for i in range(n - k + 1)]; return len(set(grams)) / max(len(grams), 1)
    counts = np.bincount(np.array(t)); p = counts[counts > 0] / n
    return {"entropy": float(-(p * np.log(p)).sum()), "distinct1": distinct(1), "distinct2": distinct(2), "distinct3": distinct(3),
            "repeat_adjacent": float(np.mean(np.array(t[1:]) == np.array(t[:-1]))), "repeated2": 1 - distinct(2), "repeated3": 1 - distinct(3)}

DFA_TOKENS = DFAConfig(min_window=8, max_window_fraction=0.25, n_scales=8, min_samples=256, hard_min_samples=48)

def temporal_metrics(hs, LATE, ch_temporal):
    vals = {"H": [], "tau": [], "slope": [], "erank": [], "velocity": []}
    for li, h in zip(LATE, hs):
        sub = h[:, ch_temporal[li]]
        for c in range(sub.shape[1]):
            x = sub[:, c]
            with warnings.catch_warnings():
                warnings.simplefilter("ignore"); vals["H"].append(dfa_exponent(x, DFA_TOKENS).alpha)
            vals["tau"].append(autocorrelation_time(x))
            f, psd = welch(x - x.mean(), fs=1.0, nperseg=min(64, len(x) // 2)); mask = f > 0
            vals["slope"].append(np.polyfit(np.log10(f[mask]), np.log10(psd[mask] + 1e-12), 1)[0])
        vals["erank"].append(effective_rank(sub)); vals["velocity"].append(trajectory_velocity(sub))
    return {k: float(np.nanmean(v)) for k, v in vals.items()}

def linear_cka(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0)
    return float(np.linalg.norm(X.T @ Y) ** 2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y) + 1e-12))
def procrustes_distance(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0); U, _, Vt = np.linalg.svd(X.T @ Y); R = U @ Vt
    return float(np.linalg.norm(X - Y @ R.T) / (np.linalg.norm(X) + 1e-12))
def attention_js(a, b):
    js = []
    for h in range(a.shape[0]):
        for q in range(a.shape[1]):
            p, r = a[h, q, :q + 1], b[h, q, :q + 1]
            p = p / (p.sum() + 1e-12); r = r / (r.sum() + 1e-12); m_ = 0.5 * (p + r)
            kl = lambda u, v: float(np.sum(u * (np.log(u + 1e-12) - np.log(v + 1e-12))))
            js.append(0.5 * kl(p, m_) + 0.5 * kl(r, m_))
    return float(np.mean(js))
def pairwise_metrics(hs_i, at_i, hs_p, at_p, ch_cka):
    return {"cka_distance": float(np.mean([1 - linear_cka(a[:, ch_cka], b[:, ch_cka]) for a, b in zip(hs_i, hs_p)])),
            "procrustes_distance": float(np.mean([procrustes_distance(a[:, ch_cka], b[:, ch_cka]) for a, b in zip(hs_i, hs_p)])),
            "attention_js": float(np.mean([attention_js(a, b) for a, b in zip(at_i, at_p)]))}

TEXT = ["entropy", "distinct1", "distinct2", "distinct3", "repeat_adjacent", "repeated2", "repeated3", "intact_nll"]
VARIANTS = ["intact", "head_pruned", "weight_noise"]
all_dec, all_sens, all_perm, all_sev = [], [], [], []

def write_combined():
    dec = pd.DataFrame(all_dec); sens = pd.DataFrame(all_sens); perm = pd.DataFrame(all_perm); sev = pd.DataFrame(all_sev)
    save_df(dec, "E4v2/decomposition"); save_df(sens, "E4v2/conditional_sensitivity"); save_df(perm, "E4v2/permutation_nulls"); save_df(sev, "E4v2/severity")
    return dec, sens, perm, sev

def analyse_model(tag, e4_t, e4_p, sev_calib):
    sev = dict(sev_calib)
    for rname in VARIANTS:
        sel = e4_t[(e4_t.generation_variant == "intact") & (e4_t.replay_variant == rname)]
        sev[f"nll_replay_{rname}_on_intact_text"] = float(sel["replay_nll"].mean())
        sev[f"nonfinite_fraction_{rname}"] = float(sel["nonfinite_fraction"].mean())
    for gname in VARIANTS:
        sev[f"intact_nll_of_{gname}_text"] = float(e4_t[e4_t.generation_variant == gname].drop_duplicates("input_id")["intact_nll"].mean())
    all_sev.append(sev)
    for met in ["H", "tau", "slope", "erank", "velocity"]:
        r = both_increments(e4_t, met, TEXT, "generation_variant", "replay_variant", design_cols=("prompt",), cv_group="prompt")
        a = attribute(e4_t, met, "replay_variant", "input_id", covariates=TEXT)
        all_dec.append({"model": tag, "family": "temporal", **r.to_row(), "share_replay_variant": a.share_system, "share_input": a.share_input})
    for met in ["cka_distance", "procrustes_distance", "attention_js", "abs_dH", "abs_dslope", "abs_derank"]:
        inc = pairwise_increment(e4_p, met, TEXT, "perturbation", "generation_variant", design_cols=("prompt",))
        all_dec.append({"model": tag, "family": "pairwise", "metric": met, "dR2_perturbation": inc.delta_r2,
                        "R2_text_only": _ols_r2(e4_p.dropna(subset=[met]), f"{met} ~ " + " + ".join(TEXT)), "R2_full": inc.r2_full, "n_rows": inc.n_rows})
    for met in ["H", "tau", "slope", "erank", "velocity"]:
        for target in VARIANTS[1:]:
            all_sens.append({"model": tag, **conditional_sensitivity(e4_t, met, "input_id", "replay_variant", "intact", target).to_row()})
    base_t = TEXT + ["C(prompt)", "C(generation_variant)"]
    for met in ["H", "slope"]:
        pr = permutation_null(e4_t, met, base_t, "replay_variant", block_cols=["input_id"], n_permutations=E4_N_PERM, seed=SEED)
        all_perm.append({"model": tag, "metric": met, "term": "replay_variant", "observed": pr.observed, "null95": pr.null_95th, "p": pr.p_value})
    for met in ["cka_distance", "abs_dH"]:
        pr = permutation_null(e4_p, met, TEXT + ["C(prompt)", "C(generation_variant)"], "perturbation", block_cols=["input_id"], n_permutations=E4_N_PERM, seed=SEED)
        all_perm.append({"model": tag, "metric": met, "term": "perturbation", "observed": pr.observed, "null95": pr.null_95th, "p": pr.p_value})

for model_name in E4_MODELS:
    tag = model_name.split("/")[-1]
    t_path, p_path = out_path(f"E4v2/temporal_{tag}", "csv"), out_path(f"E4v2/pairwise_{tag}", "csv")
    if E4_REUSE_EXISTING and os.path.exists(t_path) and os.path.exists(p_path):
        print(f"\n== {tag}: reusing existing measurements")
        e4_t, e4_p = pd.read_csv(t_path), pd.read_csv(p_path)
        calib_path = out_path(f"E4v2/calibration_{tag}", "json")
        sev_calib = json.load(open(calib_path)) if os.path.exists(calib_path) else {"model": tag}
        analyse_model(tag, e4_t, e4_p, sev_calib); write_combined()
        print(f"   analysed ({time.time()-t0:.0f} s)")
        continue
    try:
        m, tok = load_lm(model_name)
    except Exception as exc:
        print(f"\n== {tag}: could not load ({type(exc).__name__}: {exc}); skipping this model")
        continue
    LATE = late_layers(m); n_embd = m.config.hidden_size
    rng4 = np.random.default_rng(SEED)
    ch_temporal = {li: np.sort(rng4.choice(n_embd, min(E4_CHANNELS_TEMPORAL, n_embd), replace=False)) for li in LATE}
    ch_cka = np.sort(rng4.choice(n_embd, min(E4_CHANNELS_CKA, n_embd), replace=False))
    print(f"\n== {tag}: layers {m.config.num_hidden_layers}, late {LATE}, heads {m.config.num_attention_heads}, device {device} ({time.time()-t0:.0f} s)")

    # severity matching: choose the noise fraction whose NLL increase on intact text is closest to head pruning's
    def mean_nll(kind, frac=None, n=min(4, len(PROMPTS))):
        vals = []
        with perturbed(m, kind, SEED, frac):
            for prompt in PROMPTS[:n]:
                ids = encode(tok, prompt)
                with torch.no_grad():
                    lg = m(ids).logits[0, :-1].float()
                vals.append(float(-torch.log_softmax(torch.nan_to_num(lg), -1).gather(1, ids[0, 1:, None]).mean()))
        return float(np.mean(vals))
    noise_fraction = E4_NOISE_FRACTION
    calib = {}
    if E4_MATCH_SEVERITY and tok is not None:
        nll0 = mean_nll("intact"); target = mean_nll("head_pruned") - nll0
        for frac in E4_NOISE_FRACTIONS:
            calib[frac] = mean_nll("weight_noise", frac) - nll0
        noise_fraction = min(calib, key=lambda f: abs(calib[f] - target))
        print(f"   severity match: head pruning dNLL={target:+.3f}; noise dNLL by fraction={ {k: round(v, 3) for k, v in calib.items()} }; chosen {noise_fraction}")
    all_sev_calib = {"model": tag, "noise_fraction_used": noise_fraction, "prune_dNLL_target": (calib and target) or np.nan, **{f"noise_dNLL_{k}": v for k, v in calib.items()}}
    save_json(all_sev_calib, f"E4v2/calibration_{tag}")

    # generation phase: each variant generates from every prompt
    sequences = {}
    for gi, gname in enumerate(VARIANTS):
        with perturbed(m, gname, SEED, noise_fraction):
            for pi, prompt in enumerate(PROMPTS):
                sequences[(pi, gname)] = generate(m, tok, encode(tok, prompt), seed=SEED * 1000 + pi * 10 + gi)
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"   generated {len(sequences)} sequences ({time.time()-t0:.0f} s)")

    # replay phase: every sequence through every variant
    temporal_rows, pair_rows = [], []
    reps_cache = {}
    for rname in VARIANTS:
        with perturbed(m, rname, SEED, noise_fraction):
            for key, ids in sequences.items():
                reps_cache[(key, rname)] = replay(m, ids, LATE)
    for (pi, gname), ids in sequences.items():
        ts = text_stats(ids); ts["intact_nll"] = reps_cache[((pi, gname), "intact")][2]
        for rname in VARIANTS:
            hs, at, nll, nonfinite = reps_cache[((pi, gname), rname)]
            temporal_rows.append({"model": tag, "input_id": f"p{pi}_{gname}", "prompt": pi, "generation_variant": gname,
                                  "replay_variant": rname, "n_tokens": int(ids.shape[1]), "replay_nll": nll,
                                  "nonfinite_fraction": nonfinite, **ts, **temporal_metrics(hs, LATE, ch_temporal)})
        hs_i, at_i, _, _ = reps_cache[((pi, gname), "intact")]
        for pname in VARIANTS[1:]:
            hs_p, at_p, _, _ = reps_cache[((pi, gname), pname)]
            ti, tp = temporal_metrics(hs_i, LATE, ch_temporal), temporal_metrics(hs_p, LATE, ch_temporal)
            pair_rows.append({"model": tag, "input_id": f"p{pi}_{gname}", "prompt": pi, "generation_variant": gname, "perturbation": pname, **ts,
                              **pairwise_metrics(hs_i, at_i, hs_p, at_p, ch_cka),
                              "abs_dH": abs(ti["H"] - tp["H"]), "abs_dslope": abs(ti["slope"] - tp["slope"]), "abs_derank": abs(ti["erank"] - tp["erank"])})
    e4_t = pd.DataFrame(temporal_rows); e4_p = pd.DataFrame(pair_rows)
    save_df(e4_t, f"E4v2/temporal_{tag}"); save_df(e4_p, f"E4v2/pairwise_{tag}")
    del reps_cache

    analyse_model(tag, e4_t, e4_p, all_sev_calib); write_combined()
    print(f"   analysed ({time.time()-t0:.0f} s)")
    del m
    if device == "cuda":
        torch.cuda.empty_cache()

dec, sens, perm, sev = write_combined()
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("\n=== E4v2 severity (NLL in nats)"); print(sev.round(3).to_string(index=False))
print("\n=== E4v2 attribution by model"); print(dec.round(3).to_string(index=False))
print("\n=== E4v2 conditional sensitivity by model"); print(sens.round(3).to_string(index=False))
print("\n=== E4v2 permutation nulls"); print(perm.round(4).to_string(index=False))
print(f"\nE4v2 done in {time.time()-t0:.0f} s")

## Bundle

Writes the parameters used and zips the results folder.

In [ ]:
# ---- Bundle ----------------------------------------------------------------------------------
save_json({"DRY_RUN": DRY_RUN, "SEED": SEED, "E2_CHAOTIC_GAINS": E2_CHAOTIC_GAINS, "E2_CHAOTIC_SEEDS": E2_CHAOTIC_SEEDS,
           "E2_CHAOTIC_N": E2_CHAOTIC_N, "E2_N_INPUTS": E2_N_INPUTS, "E2_REPLICATES": E2_REPLICATES, "E4_MODELS": E4_MODELS,
           "E4_N_PROMPTS": E4_N_PROMPTS, "E4_NEW_TOKENS": E4_NEW_TOKENS, "E4_NOISE_FRACTION": E4_NOISE_FRACTION,
           "E4_MATCH_SEVERITY": E4_MATCH_SEVERITY, "E4_FP16_ABOVE_PARAMS": E4_FP16_ABOVE_PARAMS,
           "E4_PRUNE_KEEP_FRACTION": E4_PRUNE_KEEP_FRACTION}, "parameters")
import tempfile
stage = tempfile.mkdtemp()
for sub in ("E2v2", "E4v2"):
    shutil.copytree(os.path.join(RESULTS_DIR, sub), os.path.join(stage, sub))
shutil.copy(out_path("parameters", "json"), stage)
zip_path = shutil.make_archive(ZIP_STEM, "zip", stage)
print("results folder :", RESULTS_DIR)
print("bundle         :", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    pass